<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/proteomics-multiomics/lessons/P07-L03-metabolomics-from-peaks/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/proteomics-multiomics/lessons/P07-L03-metabolomics-from-peaks/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/proteomics-multiomics/lessons/P07-L03-metabolomics-from-peaks/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/proteomics-multiomics/lessons/P07-L03-metabolomics-from-peaks/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P07-L03 · Metabolomics from raw peaks: peak picking, retention-time alignment, adduct and isotope grouping, and annotations reported at a stated confidence level

**You will build:** the chain that turns raw liquid-chromatography mass-spectrometry (LC-MS) scans
into a list of annotated metabolite features. You will write the grouping of peaks into mass
traces, the picking of chromatographic peaks, the matching and warping of retention times across
runs, the grouping of isotope and adduct peaks that belong to one compound, the step that turns a
group into a neutral mass, and an annotation that reports **every** result at a Metabolomics
Standards Initiative (MSI) confidence level. Then you will **count**, against the list of
compounds that were planted, how many features each step finds or invents and how many
annotations at each level are wrong.

**Time:** ~150 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** `P07-L02-imputation-and-batch-effects` (the habit of scoring a call set against
a planted answer key; nothing of its code is used). Python and numpy indexing. No chemistry beyond
"a molecule has a mass, and the instrument sees it with an electric charge added".

By the end you will be able to:

1. Implement mass-trace clustering by a parts-per-million gap rule and chromatographic peak
   picking with smoothing, a signal-to-noise rule and a prominence rule.
2. Implement feature matching and piecewise-linear retention-time warping across runs, and
   measure what alignment and a minimum number of runs do to the feature table.
3. Implement isotope grouping, adduct grouping and the choice of a neutral mass for a group, and
   measure how many features, groups and planted compounds there are at each step.
4. Implement an annotation that reports Metabolomics Standards Initiative levels 1 to 4 from
   mass, isotope pattern and standard retention time, and the scoring of those levels.
5. Measure, against the planted compounds, the false-annotation rate at each level, and explain
   from the counts why a bare name on a feature is a claim that needs its evidence stated.

**The data is synthetic, and the simulator is in this notebook.** A real LC-MS study has no
answer key: nobody knows which of its features are compounds, or which compound. A simulated one
does, and a false-annotation rate is a claim about an answer key. `meta.yaml` declares the data
as synthetic, nothing is downloaded, the "compounds" are invented identifiers, and nothing you
conclude about these particular counts describes a real instrument, study or software package.
What transfers is the method, and the habit of saying how a name was reached.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `P07-L02`

**This lesson assumes you can already:**

- write Python functions and loops, and index and mask numpy arrays
- read a chromatogram (intensity against retention time) and a mass spectrum (intensity against m/z) as two views of the same LC-MS data
- say what a false discovery proportion measured against a planted answer key is (P07-L02 builds the habit)

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Proposed minimum reporting standards for chemical analysis (Chemical Analysis Working Group, Metabolomics Standards Initiative)](https://pmc.ncbi.nlm.nih.gov/articles/PMC3772505/) · Sumner et al., Metabolomics 2007 (open access on PubMed Central) · *free* — the source of the four identification levels the lesson reports, and of the minimum evidence it asks for at level 1
- [LC-MS data preprocessing and analysis with xcms (package vignette)](https://bioconductor.org/packages/release/bioc/vignettes/xcms/inst/doc/xcms.html) · Philippine Louail and Johannes Rainer, Bioconductor · *free* — the peak detection, retention-time alignment and correspondence sections show the production versions of what exercises 1 to 4 build
- [CAMERA: Collection of annotation related methods for mass spectrometry data (package page)](https://bioconductor.org/packages/release/bioc/html/CAMERA.html) · Bioconductor (package by Carsten Kuhl, Ralf Tautenhahn, Hendrik Treutler and Steffen Neumann) · *free* — the documentation of a published tool that groups isotope and adduct peaks into compound spectra, the job of exercises 5 to 7

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import os
import signal
import sys
import threading
import time
import traceback
from typing import Any, Callable, NamedTuple

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

# The world. Every number that shapes it is here, so that nothing later is a hidden knob.
SEED = 20261002              # the world you study in detail; section 11 also uses the seeds after it
N_RUNS = 4                   # runs of the same sample; run 0 is the reference for retention time
N_SCANS = 300                # scans per run
RT_MAX = 15.0                # minutes: scan i is taken at i * RT_MAX / N_SCANS
MZ_LO, MZ_HI = 80.0, 800.0   # the instrument's mass range
N_PLANTED = 70               # compounds planted at random (three more are planted as traps, below)
N_TRAPS = 3                  # compounds whose [M+H]+ sits where another compound's [M+Na]+ is
N_LIBRARY = 320              # entries in the compound library the annotation searches
NOISE_PEAKS_PER_SCAN = 30    # random noise peaks per scan (a Poisson mean)
NOISE_MEDIAN, NOISE_SIGMA = 600.0, 0.6   # noise peak intensity: lognormal, this median, this log-sd
N_CONTAMINANTS = 12          # ions present in every scan at a steady level: background, not compounds
N_BUMPS = 120                # short chromatographic bumps per run that are not compounds, new every run
INSTRUMENT_FLOOR = 150.0     # the instrument reports no peak below this intensity
MZ_JITTER_PPM = 1.5          # scan-to-scan scatter of a measured m/z, in parts per million
DETECTABLE = 5.0 * NOISE_MEDIAN   # a planted ion at least this tall is one a good pipeline ought to find

# Masses, from NIST (claims.yaml): the proton, the electron, and the atoms the adducts are made of.
PROTON = 1.0072764665789
ELECTRON = 5.485799090441e-4
MASS_H, MASS_N, MASS_NA, MASS_K = 1.00782503223, 14.00307400443, 22.9897692820, 38.9637064864
ISO_DELTA = 13.00335483507 - 12.0   # the mass of carbon-13 minus carbon-12
C13_ABUNDANCE = 0.0107              # the fraction of carbon atoms that are carbon-13
# The adducts this world can put on a molecule, as the mass added to the NEUTRAL molecule M to get
# the ion that is measured (each ion has one positive charge, so m/z = M + shift).
ADDUCTS = (("[M+H]+", PROTON),
           ("[M+NH4]+", MASS_N + 4.0 * MASS_H - ELECTRON),
           ("[M+Na]+", MASS_NA - ELECTRON),
           ("[M+K]+", MASS_K - ELECTRON))
ADDUCT_NAMES = tuple(name for name, _ in ADDUCTS)
ADDUCT_SHIFT = np.array([shift for _, shift in ADDUCTS])
ADDUCT_PRESENT_P = (0.75, 0.30, 0.55, 0.15)   # chance that a compound shows each adduct at all
CLASSES = ("amine", "acid", "sugar", "nucleoside", "lipid", "alkaloid", "phenolic", "peptide")

# The analysis settings, fixed for the whole lesson so that only the choices you study vary.
CLUSTER_PPM = 5.0            # mass traces: peaks within this gap are the same ion
MIN_TRACE_POINTS = 5         # a mass trace seen in fewer scans than this is not examined
SMOOTH_WINDOW = 5            # scans in the moving average
SNR = 3.0                    # a peak must stand this many noise levels above zero
BASE_SCANS = 15              # how far either side of an apex the prominence rule looks
PROM_FRAC = 0.5              # a peak must rise at least this fraction of its height above its base
ANCHOR_RT_TOL = 0.8          # minutes: how far apart two runs' retention times may be for an anchor
N_KNOTS = 8                  # the warp is drawn through this many median anchor points
MATCH_PPM, MATCH_RT_TOL = 5.0, 0.1   # a feature in one run is the same as one in another
MIN_RUNS = 2                 # a feature must be found in at least this many runs
GROUP_PPM, GROUP_RT_TOL = 5.0, 0.08  # isotope and adduct grouping
ANNOT_PPM, ANNOT_RT_TOL, ISO_TOL = 5.0, 0.1, 0.3   # annotation: mass window, standard RT window, isotope ratio window


# How long an exercise's check may run before it counts as stuck, and how long any other cell that runs your
# code (a demonstration, a table) may: a right answer needs a small fraction of either, measured on the worked
# solution (internal/retrofit_guard.py, 10x or more). tools/verify_all.py reads both numbers.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 10


class Library(NamedTuple):
    """The compound library the annotation searches. One row per entry, in no useful order."""
    names: tuple         # identifiers such as "CMP-017": invented, not real compounds
    mass: np.ndarray     # neutral monoisotopic mass
    n_carbon: np.ndarray # carbon atoms, which sets the expected M+1 / M intensity ratio
    klass: tuple         # a compound class such as "acid"
    std_rt: np.ndarray   # retention time of an authentic standard run on this column; NaN = no standard


class Run(NamedTuple):
    """One LC-MS run as the instrument reports it: a list of centroided peaks, scan by scan."""
    scan_rt: np.ndarray  # (N_SCANS,) retention time of each scan, minutes
    scan: np.ndarray     # (peaks,) scan number of each peak, in non-decreasing order
    mz: np.ndarray       # (peaks,) measured mass-to-charge ratio
    inten: np.ndarray    # (peaks,) intensity


class World(NamedTuple):
    """The runs an analyst receives, the library, and the answer key that generated them."""
    runs: tuple
    library: Library
    planted: np.ndarray      # library indices of the compounds that are really in the sample
    ion_comp: np.ndarray     # (ions,) library index of the compound each planted ion belongs to
    ion_adduct: np.ndarray   # (ions,) index into ADDUCTS
    ion_iso: np.ndarray      # (ions,) 0 = monoisotopic, 1 = M+1, 2 = M+2
    ion_mz: np.ndarray       # (ions,) true m/z
    ion_rt: np.ndarray       # (ions,) true retention time on the reference run's time axis
    ion_height: np.ndarray   # (ions,) intensity at the apex before run-to-run factors
    ion_sigma: np.ndarray    # (ions,) width of the chromatographic peak (standard deviation), minutes
    contaminant_mz: np.ndarray
    trap_pairs: tuple        # (library index of A, of B): B's [M+H]+ has the m/z of A's [M+Na]+
    drift: tuple             # per run, the parameters of its retention-time drift
    seed: int                # the seed it was generated from


def iso_ratio(n_carbon: int, k: int) -> float:
    """Expected intensity of the k-th isotope peak over the monoisotopic one: binomial in carbon-13."""
    p = C13_ABUNDANCE
    return math.comb(n_carbon, k) * (p / (1.0 - p)) ** k


def _drift(world_or_params, run: int, t):
    """Given. The run's retention time at reference time t (answer key; no analyst sees this)."""
    a, b, c, f, ph = world_or_params[run]
    return t + a + b * t / RT_MAX + c * np.sin(2.0 * math.pi * f * t / RT_MAX + ph)


def generate_world(seed: int = SEED, n_runs: int = N_RUNS) -> World:
    """Given to you. Simulate LC-MS runs of one sample, a compound library and the answer key.

    Planted compounds have a neutral mass, a carbon count (which sets their isotope pattern), a
    class, a retention time and a peak width. Each shows one to four adducts, and each adduct three
    isotope peaks, as Gaussian chromatographic peaks. Every run drifts in retention time by a
    smooth amount, scatters every measured m/z a little, scales every compound by its own factor,
    and misses a compound entirely now and then. Added to every run: random noise peaks, steady
    background ions, and short bumps that are not compounds. The library holds the planted
    compounds, relatives of them (isomers, species whose [M+H]+ sits where another's adduct does,
    near-isobars) and compounds that are not in the sample at all; half of its entries have an
    authentic standard whose retention time was measured on this column.
    """
    rng = np.random.default_rng(seed)
    n0 = N_PLANTED
    mass = list(rng.uniform(110.0, 600.0, n0))
    nc = [int(v) for v in np.clip(np.round(np.array(mass) / rng.uniform(16.0, 30.0, n0)), 4, 45)]
    klass = [int(v) for v in rng.integers(0, len(CLASSES), n0)]
    rt = list(rng.uniform(0.8, 13.5, n0))
    shift_na_h = ADDUCT_SHIFT[2] - ADDUCT_SHIFT[0]
    pairs = []
    for t in range(N_TRAPS):                 # trap B has the mass of A plus (Na - H); it elutes elsewhere
        sign = 1.0 if rng.random() < 0.5 else -1.0
        gap = sign * rng.uniform(1.5, 3.0)
        rt_b = rt[t] + gap if 0.8 <= rt[t] + gap <= 13.5 else rt[t] - gap
        mass.append(mass[t] + shift_na_h); nc.append(nc[t])
        klass.append(int(rng.integers(0, len(CLASSES)))); rt.append(rt_b)
        pairs.append((t, n0 + t))
    n_planted = len(mass)
    kinds = ("isomer", "na", "nh4", "k", "h2", "near")
    for c in range(n_planted):               # library-only relatives of every planted compound
        for _ in range(int(rng.poisson(1.3))):
            kind = kinds[int(rng.integers(0, len(kinds)))]
            m, n, cl = mass[c], nc[c], klass[c]
            if kind == "isomer":             # same formula: same mass, same carbon count
                cl = cl if rng.random() < 0.7 else int(rng.integers(0, len(CLASSES)))
            elif kind == "near":             # a different formula a few ppm away
                m *= 1.0 + rng.choice([-1.0, 1.0]) * rng.uniform(2.0, 8.0) * 1e-6
                n = max(4, n + int(rng.choice([-3, -2, -1, 1, 2, 3])))
                cl = int(rng.integers(0, len(CLASSES)))
            else:                            # its [M'+H]+ coincides with an adduct of the planted compound
                m += {"na": shift_na_h, "nh4": ADDUCT_SHIFT[1] - ADDUCT_SHIFT[0],
                      "k": ADDUCT_SHIFT[3] - ADDUCT_SHIFT[0], "h2": 2.0 * MASS_H}[kind]
                cl = int(rng.integers(0, len(CLASSES)))
            mass.append(m); nc.append(n); klass.append(cl); rt.append(float(rng.uniform(0.8, 13.5)))
    while len(mass) < N_LIBRARY:             # compounds that are not in the sample at all
        m = float(rng.uniform(110.0, 600.0))
        mass.append(m); nc.append(int(np.clip(round(m / rng.uniform(16.0, 30.0)), 4, 45)))
        klass.append(int(rng.integers(0, len(CLASSES)))); rt.append(float(rng.uniform(0.8, 13.5)))
    total = len(mass)
    perm = rng.permutation(total)            # the library's order must not reveal who is planted
    inv = np.empty(total, dtype=int)
    inv[perm] = np.arange(total)
    has_std = rng.random(total) < 0.5
    rt_a = np.array(rt)[perm]
    std_rt = np.where(has_std, rt_a + rng.normal(0.0, 0.01, total), np.nan)
    lib = Library(tuple(f"CMP-{i:03d}" for i in range(total)), np.array(mass)[perm],
                  np.array(nc)[perm], tuple(CLASSES[k] for k in np.array(klass)[perm]), std_rt)
    planted = inv[:n_planted]                # planted compound j is library entry inv[j]
    pairs = tuple((int(inv[a]), int(inv[b])) for a, b in pairs)
    weights = rng.uniform(0.2, 1.0, (n_planted, len(ADDUCTS)))
    present = rng.random((n_planted, len(ADDUCTS))) < np.array(ADDUCT_PRESENT_P)
    for j in range(n_planted):
        if j < N_TRAPS:                      # a trap's parent shows both [M+H]+ and [M+Na]+
            present[j, 0] = present[j, 2] = True
        if not present[j].any():
            present[j, int(rng.integers(0, len(ADDUCTS)))] = True
    base = 10.0 ** rng.uniform(3.4, 6.0, n_planted)
    sigma = rng.uniform(0.10, 0.20, n_planted)    # peak width (standard deviation), minutes
    ions = [(int(planted[j]), a, k, mass[j] + ADDUCT_SHIFT[a] + k * ISO_DELTA, rt[j],
             base[j] * weights[j, a] * iso_ratio(nc[j], k), sigma[j], j)
            for j in range(n_planted) for a in range(len(ADDUCTS)) if present[j, a] for k in range(3)]
    ion_comp, ion_adduct, ion_iso, ion_mz, ion_rt, ion_h, ion_sig, ion_j = (np.array(col) for col in zip(*ions))
    cont_mz = rng.uniform(100.0, 700.0, N_CONTAMINANTS)
    cont_level = 10.0 ** rng.uniform(3.3, 4.0, N_CONTAMINANTS)
    dt = RT_MAX / N_SCANS
    scan_rt = np.arange(N_SCANS) * dt
    runs, drifts = [], []
    for r in range(n_runs):
        if r == 0:
            drifts.append((0.0, 0.0, 0.0, 1.0, 0.0))
        else:
            drifts.append((rng.uniform(-0.25, 0.25), rng.uniform(-0.3, 0.3), rng.uniform(0.03, 0.08),
                           rng.uniform(1.0, 2.5), rng.uniform(0.0, 2.0 * math.pi)))
        cal = rng.normal(0.0, 1.0) * 1e-6             # this run's mass-calibration offset
        run_factor = np.exp(rng.normal(0.0, 0.25, n_planted))
        missing = rng.random(n_planted) < 0.08
        S, MZ, IN = [], [], []

        def add(centre, sg, height, mz_true):
            lo = max(0, int(math.floor((centre - 4.0 * sg) / dt)))
            hi = min(N_SCANS, int(math.ceil((centre + 4.0 * sg) / dt)) + 1)
            if hi <= lo:
                return
            s = np.arange(lo, hi)
            g = height * np.exp(-0.5 * ((scan_rt[s] - centre) / sg) ** 2) * np.exp(rng.normal(0.0, 0.05, len(s)))
            keep = g >= INSTRUMENT_FLOOR
            s, g = s[keep], g[keep]
            S.append(s); IN.append(g)
            MZ.append(mz_true * (1.0 + cal + rng.normal(0.0, MZ_JITTER_PPM * 1e-6, len(s))))

        for q in range(len(ion_mz)):
            j = int(ion_j[q])
            if missing[j]:
                continue
            add(float(_drift(drifts, r, ion_rt[q])), ion_sig[q],
                ion_h[q] * run_factor[j] * math.exp(rng.normal(0.0, 0.05)), ion_mz[q])
        for c in range(N_CONTAMINANTS):               # steady background ions, present in every scan
            S.append(np.arange(N_SCANS)); IN.append(cont_level[c] * np.exp(rng.normal(0.0, 0.2, N_SCANS)))
            MZ.append(cont_mz[c] * (1.0 + cal + rng.normal(0.0, MZ_JITTER_PPM * 1e-6, N_SCANS)))
        for _ in range(N_BUMPS):                      # bumps that are not compounds: new in every run
            add(rng.uniform(0.5, RT_MAX - 0.5), rng.uniform(0.10, 0.20), 10.0 ** rng.uniform(3.0, 3.7),
                rng.uniform(MZ_LO, MZ_HI))
        counts = rng.poisson(NOISE_PEAKS_PER_SCAN, N_SCANS)
        n_noise = int(counts.sum())
        S.append(np.repeat(np.arange(N_SCANS), counts)); MZ.append(rng.uniform(MZ_LO, MZ_HI, n_noise))
        IN.append(NOISE_MEDIAN * np.exp(rng.normal(0.0, NOISE_SIGMA, n_noise)))
        S, MZ, IN = np.concatenate(S), np.concatenate(MZ), np.concatenate(IN)
        order = np.argsort(S, kind="stable")
        runs.append(Run(scan_rt, S[order], MZ[order], IN[order]))
    return World(tuple(runs), lib, planted, ion_comp.astype(int), ion_adduct.astype(int), ion_iso.astype(int),
                 ion_mz, ion_rt, ion_h, ion_sig, cont_mz, pairs, tuple(drifts), seed)


def connected_labels(n: int, pairs) -> np.ndarray:
    """Given to you. Label the connected groups of `n` items joined by `pairs` of indices.

    Every item gets the SMALLEST index in its group as its label, so the answer does not depend on
    the order the pairs come in. Items in no pair are labelled with their own index.
    """
    parent = list(range(n))

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i, j in pairs:
        a, b = find(int(i)), find(int(j))
        if a != b:
            parent[max(a, b)] = min(a, b)
    return np.array([find(i) for i in range(n)], dtype=int)


def noise_level(run: Run) -> float:
    """Given to you. The run's noise level: the median intensity of all its peaks.

    Most peaks in a scan are noise, so the median sits near the noise. It sits a little above it,
    because signal peaks are in the mix; section 2 prints both.
    """
    return float(np.median(run.inten))


def build_traces(run: Run, labels: np.ndarray, min_points: int = MIN_TRACE_POINTS):
    """Given to you. One extracted-ion chromatogram per cluster of peaks with at least `min_points` peaks.

    Returns `(mz, ys)`: the intensity-weighted mean m/z of each trace, and for each one an array of
    length N_SCANS holding the summed intensity in every scan (0 where the trace has no peak).
    """
    order = np.argsort(labels, kind="stable")
    cut = np.nonzero(np.diff(labels[order]))[0] + 1
    mzs, ys = [], []
    for idx in np.split(order, cut):
        if len(idx) < min_points:
            continue
        w = run.inten[idx]
        mzs.append(float((run.mz[idx] * w).sum() / w.sum()))
        ys.append(np.bincount(run.scan[idx], weights=w, minlength=len(run.scan_rt)))
    return np.array(mzs), ys


def tag_truth(world: World, mz: np.ndarray, rt: np.ndarray, ppm: float = 10.0, rt_tol: float = 0.15) -> np.ndarray:
    """Given (answer key). For each feature, the index of the planted ion it is, or -1 if it is none.

    A feature is a planted ion when its m/z is within `ppm` of the ion's and its retention time,
    on the reference run's axis, is within `rt_tol` minutes; the nearest in time wins.
    """
    out = -np.ones(len(mz), dtype=int)
    for i in range(len(mz)):
        c = np.nonzero(np.abs(world.ion_mz - mz[i]) <= ppm * 1e-6 * mz[i])[0]
        if len(c):
            d = np.abs(world.ion_rt[c] - rt[i])
            if d.min() <= rt_tol:
                out[i] = c[int(np.argmin(d))]
    return out


def _show(fig: "matplotlib.figure.Figure") -> None:
    """Display a figure in Jupyter, or close it cleanly in a headless script run."""
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("cluster_mz",),
    "exercise 2": ("detect_peaks",),
    "exercise 3": ("match_features",),
    "exercise 4": ("warp_rt",),
    "exercise 5": ("group_isotopes",),
    "exercise 6": ("group_adducts",),
    "exercise 7": ("neutral_mass",),
    "exercise 8": ("annotate",),
    "exercise 9": ("score_annotations",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run
_UPTO = {i: tuple(f"exercise {k}" for k in range(1, i + 1)) for i in range(1, 10)}
# The heavy demos (a whole world, then several of them) run when the file is run as a notebook or a
# script, and are skipped when the autograder imports it, so grading stays quick.
_RUN_DEMOS = __name__ == "__main__"


_MEMO: dict = {}


def _memo(key: tuple, dep_names: tuple[str, ...], make: Callable[[], Any]) -> Any:
    """Compute `make()` once per (key, the functions named in `dep_names` as they are NOW).

    The demos reuse the peak tables and groups that your functions made. If you edit one of those
    functions and re-run its cell, the new function object is a new key, so nothing stale is reused.
    The key holds the function objects themselves, never their `id()`: Python hands a freed object's
    id to the next new one, so a key of ids can match a function you replaced two edits ago.
    """
    full = (key, tuple(globals()[name] for name in dep_names))
    if full not in _MEMO:
        _MEMO[full] = make()
    return _MEMO[full]


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (match_features)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the message. Why it is this careful: whoever runs the notebook may
# already have an alarm running (tools/grade.py stops every rubric row after 45 s and repeats
# its alarm every 0.25 s; a harness may have an alarm that ends the process). Earlier guards
# switched that alarm off on the way out (`setitimer(ITIMER_REAL, 0)`), so the NEXT hang ran for
# ever. This one saves the running alarm, lets whichever deadline comes first win (handing the
# signal to the outer handler when it is the outer's turn), and on the way out re-arms the outer
# alarm with the time it had left and its interval. tools/verify_all.py fails a lesson whose
# guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s in case a bare
    `except:` does. `_time_limit` turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever, or work grows far faster than the data does (a Python "
               "loop over every pair of peaks instead of one array operation per row)")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = (),
         limit: int = 0) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        # 2 s for an exercise check (its calls are tiny, so a loop that never ends is stopped before
        # it can append its way to gigabytes); demos run whole worlds and get longer.
        with _time_limit(limit or (_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS), label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _demo(label: str, fn: Callable[[], None], needs: tuple[str, ...] = (), limit: int = 0) -> None:
    """A demo cell: guarded like a check, and skipped when the file is imported by the grader."""
    if _RUN_DEMOS:
        _try(label, fn, needs, limit)



In [ ]:
def _given_checks() -> None:
    """The given helpers that everything downstream trusts, checked against closed forms."""
    assert connected_labels(5, [(3, 1), (4, 3)]).tolist() == [0, 1, 2, 1, 1], "connected_labels: broken"
    assert connected_labels(4, []).tolist() == [0, 1, 2, 3], "connected_labels: an unlinked item is its own group"
    assert abs(iso_ratio(10, 1) - 10 * C13_ABUNDANCE / (1 - C13_ABUNDANCE)) < 1e-12, "iso_ratio: M+1 / M is n p / (1 - p)"
    assert iso_ratio(10, 0) == 1.0, "iso_ratio: M / M is 1"
    na_minus_h = ADDUCT_SHIFT[2] - ADDUCT_SHIFT[0]
    assert abs(na_minus_h - (MASS_NA - MASS_H)) < 2e-6, "adduct shifts: Na+ minus H+ is the Na atom minus the H atom"
    w = generate_world(SEED + 99, n_runs=2)
    again = generate_world(SEED + 99, n_runs=2)
    assert all(np.array_equal(a.mz, b.mz) and np.array_equal(a.inten, b.inten) for a, b in zip(w.runs, again.runs)), \
        "generate_world must be a pure function of its seed"
    print("the given helpers agree with the closed forms, and the world is reproducible from its seed")


_try("given: helpers", _given_checks)

## 1. The problem in one page

A liquid-chromatography mass spectrometer does two things in a row. The **column** separates a
mixture in time, so each compound comes off at its own **retention time**. The **mass
spectrometer** then takes scan after scan, and a scan is a list of peaks, each with a
**mass-to-charge ratio** (m/z) and an intensity. Pile the scans up and a compound appears as a
small hill in time at one m/z. That is the raw material, and three facts make it hard.

- **A compound is not one peak.** A molecule M is measured as an *ion*: M with a charged particle
  attached, an **adduct**. Which adduct is a matter of chemistry in the source, so one compound
  can show several peaks, at m/z = M + a different amount for each. And every ion has **isotope
  peaks** a little above it, because some of its carbon atoms are carbon-13. A "feature list"
  straight from peak picking counts ions and isotopes, not compounds.
- **Time drifts.** The same compound elutes at a slightly different retention time in every run,
  and the shift can exceed the width of its own peak, so peaks from different runs cannot be
  compared until the time axes are lined up.
- **A name is a claim.** Compounds with the same formula (isomers) have the same mass, so a
  measured mass alone cannot tell them apart; the library here has some, and others a few parts
  per million apart. And an adduct mistaken for a molecule can match the wrong compound exactly.
  The Metabolomics Standards Initiative (MSI) asks authors to say how rigorously each metabolite
  was identified (`claims.yaml`), and this lesson reports every annotation at one of its levels
  instead of as a bare name.

The plan. Section 2 sets up the world. Sections 3 and 4 find the chromatographic peaks. Section 5
lines the runs up. Sections 6 to 8 group the peaks that belong together and choose a neutral
mass. Sections 9 to 11 annotate and score, and repeat everything on several simulated worlds so
you can see how much of what you saw was the luck of one seed.

## 2. The world you will study

`generate_world` (given, above) simulates several runs of one sample and keeps the answer key
beside them. Read what it made before you analyse it. The cell below prints only numbers computed
from the world; nothing in the prose is typed in from a run.

In [ ]:
WORLD = generate_world(SEED)


def _describe_world(w: World) -> None:
    lib = w.library
    print(f"{len(w.runs)} runs of {N_SCANS} scans over {RT_MAX:g} min; peaks per run: "
          + ", ".join(str(len(r.mz)) for r in w.runs))
    print("median peak intensity per run (used below as the noise level): "
          + ", ".join(f"{noise_level(r):.0f}" for r in w.runs)
          + f"; the noise peaks alone have a median of {NOISE_MEDIAN:.0f}")
    seen = w.ion_height >= DETECTABLE
    by_adduct = ", ".join(f"{int((w.ion_adduct == a).sum())} {ADDUCT_NAMES[a]}" for a in range(len(ADDUCTS)))
    print(f"planted: {len(w.planted)} compounds showing {len(w.ion_mz)} ions in all ({by_adduct}); "
          f"{int(seen.sum())} of them reach {DETECTABLE:.0f} at the apex, "
          f"{int((seen & (w.ion_iso == 0)).sum())} of those monoisotopic")
    print(f"library: {len(lib.names)} entries, {int((~np.isnan(lib.std_rt)).sum())} with an authentic standard "
          f"measured on this column; {len(w.planted)} of the entries are really in the sample")
    print(f"traps: {len(w.trap_pairs)} pairs of compounds where B's [M+H]+ has the m/z of A's [M+Na]+, "
          "eluting at different times")
    print(f"a planted ion's chromatographic peak has a standard deviation of {w.ion_sigma.min():.2f} to {w.ion_sigma.max():.2f} min")
    print(f"background in every run: {N_CONTAMINANTS} steady ions and {N_BUMPS} short bumps that are not compounds")
    for r in range(1, len(w.runs)):
        t = np.linspace(0.0, RT_MAX, 301)
        d = _drift(w.drift, r, t) - t
        print(f"  run {r} drifts from the reference by {d.min():+.2f} to {d.max():+.2f} min")


_describe_world(WORLD)

Three things to notice before you write any code. The library's order tells you nothing about who
is planted, and only some of its entries have a standard, so many compounds that are in the sample
have none. Some library entries are relatives of planted compounds on purpose: the same formula
(an isomer), a mass that is the planted one plus the difference between two adducts, and a mass a
few parts per million away. And compare each run's drift with the peak widths printed above it:
until you line the runs up, "the same feature in another run" may be further away than a peak is wide.

How this world was made, for the record: the background bumps, and peaks wider than in the first
draft, were added after a single trial run of this seed showed that without them peak picking had
nothing to get wrong (it found only compounds at every signal-to-noise bar). Both were fixed before
any exercise or table here was written; the seed is the authoring date and was never searched, and
section 11 reruns the whole chain on the next seeds.

## 3. Exercise 1 — `cluster_mz()`

An ion shows up in every scan while it elutes, each time at almost the same m/z: the measurement
scatters by a few parts per million (ppm). To follow it through time you first need to know which
peaks, from all the scans, are the same ion. The standard trick is to ignore time at first and
cluster all the peaks of a run by m/z alone. Each cluster, summed scan by scan, is an
**extracted-ion chromatogram**, a **mass trace**.

**Single linkage.** Sort the peaks by m/z. Two neighbours in that order, `a <= b`, belong together
when the gap `b - a` is at most `ppm * 1e-6 * a`, measured against the *lower* value. A cluster
keeps growing while each next gap is small enough, so a chain of small gaps is one cluster even
when its two ends are further apart than the tolerance. Gaps that sit exactly on the tolerance
are not graded: a float lands on either side of them.

<details><summary>💡 Hint 1 — what to think about</summary>

Sorting is the whole idea: once the values are in order, two peaks are in the same cluster
exactly when every neighbouring gap between them is small, so you only ever compare neighbours.
What number labels a cluster, and how do you get the labels back to the order of the input?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Get the order that sorts the values, and the sorted values. Mark every position where the gap to
the previous sorted value is larger than the tolerance (of the previous value): that starts a new
cluster. A running count of those marks, from the lowest value up, is the label in sorted order.
Write the labels back into a new array at the positions the sort came from. Reject an `mz` that is
not one-dimensional or has a non-positive or non-finite value, and a `ppm` that is not above 0,
with `ValueError`; empty input gives an empty integer array.
</details>

In [ ]:
def cluster_mz(mz: np.ndarray, ppm: float) -> np.ndarray:
    """Cluster m/z values by single linkage with a gap tolerance in parts per million.

    Sorted neighbours `a <= b` are in one cluster when `b - a <= ppm * 1e-6 * a`. Labels count
    0, 1, 2 ... from the lowest cluster upward and are returned in the order of the input.
    Equal values share a cluster. Raise `ValueError` if `mz` is not a one-dimensional array of
    positive finite numbers or `ppm` is not above 0; empty input gives an empty integer array.

    Example:
        >>> cluster_mz(np.array([200.0010, 100.0, 100.0004, 200.0]), ppm=10).tolist()
        [1, 0, 0, 1]

    Returns:
        An integer array with one label per input value.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_cluster() -> None:
    got = cluster_mz(np.array([200.0010, 100.0, 100.0004, 200.0]), ppm=10)
    assert got.tolist() == [1, 0, 0, 1], (
        f"expected [1, 0, 0, 1], got {got.tolist()}. Labels count up from the LOWEST m/z and go back in the "
        "order of the input, not the sorted order")
    chain = cluster_mz(np.array([1000.000, 1000.004, 1000.008]), ppm=5)
    assert chain.tolist() == [0, 0, 0], (
        f"three peaks 4 ppm apart in a row are one cluster, got {chain.tolist()}. Compare each peak with its "
        "NEXT neighbour (single linkage), not with the first peak of the cluster")
    split = cluster_mz(np.array([1000.0, 1000.012]), ppm=5)
    assert split.tolist() == [0, 1], f"a gap of 12 ppm is wider than 5 ppm: two clusters, got {split.tolist()}"
    wide = cluster_mz(np.array([4.0, 5.2]), ppm=250000)
    assert wide.tolist() == [0, 1], (
        f"a 25% tolerance measured against the lower value 4.0 is 1.0, and the gap is 1.2: two clusters, "
        f"got {wide.tolist()}")
    same = cluster_mz(np.array([300.0, 100.0, 100.0]), ppm=5)
    assert same.tolist() == [1, 0, 0], (
        f"two peaks at exactly the same m/z are one cluster: expected [1, 0, 0], got {same.tolist()}. A gap of "
        "zero is within any tolerance; a loop that skips equal values must still move on to the next one")
    assert len(cluster_mz(np.array([]), ppm=5)) == 0, "empty input gives an empty array"
    for bad_mz, bad_ppm in ((np.array([[100.0, 101.0]]), 5), (np.array([100.0, -1.0]), 5),
                            (np.array([100.0, np.nan]), 5), (np.array([100.0]), 0)):
        try:
            cluster_mz(bad_mz, bad_ppm)
        except ValueError:
            pass
        else:
            raise AssertionError(f"cluster_mz({bad_mz!r}, {bad_ppm}) must raise ValueError")
    print("exercise 1 looks right — peaks clustered by m/z with a ppm gap rule")

In [ ]:
_try("exercise 1", _check_cluster)

Run the clustering on the first run and see what it gives. A mass trace needs a minimum number of
peaks to be worth examining (`MIN_TRACE_POINTS`): most noise peaks stand alone, and a real ion is
seen scan after scan. The cell then changes the tolerance, because it is the first of the choices
that decide everything downstream: too narrow splits one ion into several traces, and too wide
welds neighbouring ions together. (The answer key counts a planted ion as in a trace when its true
m/z lies inside the range of m/z that the trace's peaks were measured at, widened a little.)

In [ ]:
def _traces_demo() -> None:
    run = WORLD.runs[0]
    seen = np.nonzero(WORLD.ion_height >= DETECTABLE)[0]
    print(f"{'ppm':>6} {'clusters':>9} {'traces':>7} {'ions with >1 trace':>19} {'traces holding >1 ion':>22}")
    for ppm in (1.0, CLUSTER_PPM, 50.0, 500.0):
        labels = cluster_mz(run.mz, ppm)
        order = np.argsort(labels, kind="stable")
        members = [idx for idx in np.split(order, np.nonzero(np.diff(labels[order]))[0] + 1)
                   if len(idx) >= MIN_TRACE_POINTS]                  # the traces build_traces keeps
        lo = np.array([run.mz[idx].min() for idx in members])
        hi = np.array([run.mz[idx].max() for idx in members])
        # an ion is in a trace when its true m/z lies within the m/z range the trace's peaks were
        # measured at, widened by 5 ppm each side
        inside = ((WORLD.ion_mz[seen][:, None] >= lo[None, :] * (1.0 - 5e-6))
                  & (WORLD.ion_mz[seen][:, None] <= hi[None, :] * (1.0 + 5e-6)))
        print(f"{ppm:6g} {int(labels.max()) + 1:9d} {len(members):7d} {int((inside.sum(1) > 1).sum()):19d} "
              f"{int((inside.sum(0) > 1).sum()):22d}")
    mzs, _ = build_traces(run, cluster_mz(run.mz, CLUSTER_PPM))
    found = sum(bool(np.any(np.abs(mzs - WORLD.ion_mz[q]) <= 10e-6 * WORLD.ion_mz[q])) for q in seen)
    other = [m for m in mzs if not np.any(np.abs(WORLD.ion_mz[seen] - m) <= 10e-6 * m)]
    weaker = sum(bool(np.any(np.abs(WORLD.ion_mz - m) <= 10e-6 * m)) for m in other)
    print(f"at {CLUSTER_PPM:g} ppm, {found} of the {len(seen)} planted ions at least {DETECTABLE:.0f} tall have a mass trace; "
          f"of the other {len(other)} traces, {weaker} match a weaker planted ion and {len(other) - weaker} match none (background, bumps)")


_demo("mass traces", _traces_demo, needs=_UPTO[1])

Read the table. The tolerance should sit well above the scatter of one ion's measured m/z and well
below the gap between neighbouring ions. Look at the last column at a sensible tolerance: when it
is above zero, two planted ions with (almost) the same m/z that elute at different times share one
mass trace. The traps of section 2 are built that way and chance can do it too; telling such ions
apart in time is the next exercise's job.

## 4. Exercise 2 — `detect_peaks()`

A mass trace is an intensity for each scan. A chromatographic peak is a hill in it, and picking
the hills is where most false features are born. You will write the simplest version that is not
naive, in four steps on one trace `y`:

1. **Smooth.** `s[k]` is the mean of `y` over the `window` scans centred on `k`; near the two ends
   the window shrinks to the scans that exist (so the mean is over fewer values, not padded with 0).
2. **Local maxima.** An apex is an interior scan `k` (never the first or last) with
   `s[k] > s[k-1]` and `s[k] >= s[k+1]`: on a flat top the first scan of the plateau is the apex.
3. **Signal-to-noise.** Keep it when `s[k] >= snr * noise`, the threshold included.
4. **Prominence.** Look `base_scans` scans each way. The left base is the smallest `s` from
   `k - base_scans` to `k`, the right base the smallest from `k` to `k + base_scans` (cut at the
   ends of the trace). Keep the apex when `s[k] - max(left base, right base) >= prom_frac * s[k]`.

The last rule is what rejects a steady background ion: its trace is high but flat, so no apex rises
far above its base, and the signal-to-noise rule alone would pass it.

<details><summary>💡 Hint 1 — what to think about</summary>

Do the four steps in order and keep the smoothed trace for all of them: the rules are about `s`,
not about `y`. For the smoothing, what is the count of scans in the window at each position, and
how do you get a window sum quickly (think running totals)? The two bases are separate minima over
separate slices; the *higher* of the two is what the apex has to clear.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Build the smoothed array first, with window sums from a cumulative sum divided by the number of
scans actually in each window. Then visit each interior position: skip it unless it is a local
maximum, then unless it clears the signal-to-noise bar, then unless its prominence clears the
fraction of its own height. Collect the positions and return them in increasing order as an
integer array. A `window` that is not a positive odd integer, a non-finite `y`, a `y` that is not
one-dimensional, a non-positive `snr` or `noise`, a `base_scans` below 1 and a `prom_frac`
outside 0 to 1 are `ValueError`s.
</details>

In [ ]:
def detect_peaks(y: np.ndarray, window: int = 5, snr: float = 3.0, noise: float = 1.0,
                 base_scans: int = 15, prom_frac: float = 0.5) -> np.ndarray:
    """Positions of the chromatographic peaks in one trace: smooth, local maximum, S/N, prominence.

    Apexes are interior positions `k` of the moving-average-smoothed trace `s` (window shrinking
    at the ends) with `s[k] > s[k-1]` and `s[k] >= s[k+1]`, `s[k] >= snr * noise`, and
    `s[k] - max(min(s[k-base_scans:k+1]), min(s[k:k+base_scans+1])) >= prom_frac * s[k]` (slices
    cut at the ends of the trace). Raise `ValueError` for a `y` that is not one-dimensional and
    finite, a `window` that is not a positive odd integer, a non-positive `snr` or `noise`, a
    `base_scans` below 1, or a `prom_frac` outside 0 to 1.

    Example:
        >>> y = np.array([0, 0, 1, 4, 8, 4, 1, 0, 0], dtype=float)
        >>> detect_peaks(y, window=1, snr=2.0, noise=1.0, base_scans=4, prom_frac=0.5).tolist()
        [4]

    Returns:
        An integer array of apex positions in increasing order (empty when there are none).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_detect() -> None:
    y = np.array([0, 0, 1, 4, 8, 4, 1, 0, 0], dtype=float)
    got = detect_peaks(y, window=1, snr=2.0, noise=1.0, base_scans=4, prom_frac=0.5)
    assert got.tolist() == [4], f"one clear hill: expected [4], got {got.tolist()}"
    two = np.array([0, 3, 9, 3, 0, 0, 0, 5, 12, 5, 0], dtype=float)
    got = detect_peaks(two, window=1, snr=2.0, noise=1.0, base_scans=4, prom_frac=0.5)
    assert got.tolist() == [2, 8], f"two separate hills: expected [2, 8], got {got.tolist()}"
    flat = np.array([8, 9, 8, 9, 8, 9, 8, 9, 8], dtype=float)
    got = detect_peaks(flat, window=1, snr=2.0, noise=1.0, base_scans=4, prom_frac=0.5)
    assert got.tolist() == [], (
        f"a high, flat, jittering trace has no apex that rises far above its base: expected [], got "
        f"{got.tolist()}. The prominence rule is missing or compares with the wrong base")
    low = detect_peaks(y, window=1, snr=2.0, noise=5.0, base_scans=4, prom_frac=0.5)
    assert low.tolist() == [], f"an apex of 8 is under 2 x 5: expected [], got {low.tolist()}"
    spike = np.array([0, 0, 0, 10, 0, 0, 0], dtype=float)
    got = detect_peaks(spike, window=3, snr=1.0, noise=1.0, base_scans=3, prom_frac=0.0)
    assert got.tolist() == [2], (
        f"smoothed with a window of 3 the spike becomes a plateau of three equal values; the apex is the FIRST "
        f"scan of the plateau, position 2; got {got.tolist()}")
    edge = np.array([9, 5, 2, 1, 0, 0], dtype=float)
    assert detect_peaks(edge, window=1, snr=1.0, noise=1.0, base_scans=3, prom_frac=0.0).tolist() == [], (
        "the first and last scans are never apexes: a trace that is already falling at its start has no peak")
    try:
        detect_peaks(y, window=4)
    except ValueError:
        pass
    else:
        raise AssertionError("an even window must raise ValueError")
    print("exercise 2 looks right — smoothing, local maxima, signal-to-noise and prominence")

In [ ]:
_try("exercise 2", _check_detect)

Now pick the peaks of every mass trace in the first run, and ask the answer key two questions of the
list: what share of what it found is a planted ion (**precision**), and what share of the planted
ions that are tall enough to find did it find (**recall**). Both depend on `snr` and on whether the
prominence rule is on, and the table shows how.

In [ ]:
def run_peak_table(run: Run, noise: float | None = None, snr: float = SNR, prom_frac: float = PROM_FRAC,
                   ppm: float = CLUSTER_PPM):
    """Given. Chromatographic peaks of one run as arrays `(mz, rt, height)`, using your exercises 1 and 2."""
    noise = noise_level(run) if noise is None else noise
    mzs, ys = build_traces(run, cluster_mz(run.mz, ppm))
    out_mz, out_rt, out_h = [], [], []
    for m, y in zip(mzs, ys):
        for k in detect_peaks(y, SMOOTH_WINDOW, snr, noise, BASE_SCANS, prom_frac):
            out_mz.append(m)
            out_rt.append(float(run.scan_rt[k]))
            out_h.append(float(y[max(0, k - 1):k + 2].max()))
    return np.array(out_mz), np.array(out_rt), np.array(out_h)


def _peak_scores(w: World, table) -> tuple[int, float, float]:
    """(peaks, precision, recall) of a reference-run peak table against the planted ions."""
    tags = tag_truth(w, table[0], table[1])
    seen = set(np.nonzero(w.ion_height >= DETECTABLE)[0].tolist())
    found = set(tags[tags >= 0].tolist())
    return len(tags), float((tags >= 0).mean()) if len(tags) else float("nan"), len(found & seen) / len(seen)


def _peaks_demo() -> None:
    run = WORLD.runs[0]
    print(f"noise level {noise_level(run):.0f}; the planted ions that count are at least {DETECTABLE:.0f} tall")
    print(f"{'snr':>5} {'prominence':>11} {'peaks':>6} {'precision':>10} {'recall':>7}")
    for snr, frac in ((1.0, PROM_FRAC), (2.0, PROM_FRAC), (SNR, PROM_FRAC), (5.0, PROM_FRAC),
                      (8.0, PROM_FRAC), (SNR, 0.0)):
        n, prec, rec = _peak_scores(WORLD, run_peak_table(run, snr=snr, prom_frac=frac))
        print(f"{snr:5g} {frac:11g} {n:6d} {100 * prec:9.1f}% {100 * rec:6.1f}%")


_demo("peak picking", _peaks_demo, needs=_UPTO[2])

Read down the first five rows, where only the signal-to-noise bar changes: which column falls as
the bar rises, and what does the other one do? There is no setting that is right; there is the setting whose
cost you can state. The last row turns the prominence rule off and keeps the signal-to-noise bar where
it was, which is how a steady background ion can turn into a long list of "peaks". The figure
shows one of each.

In [ ]:
def _peaks_figure() -> None:
    run = WORLD.runs[0]
    mzs, ys = build_traces(run, cluster_mz(run.mz, CLUSTER_PPM))
    noise = noise_level(run)
    top = int(np.argmax(WORLD.ion_height * (WORLD.ion_iso == 0)))
    pick = int(np.argmin(np.abs(mzs - WORLD.ion_mz[top])))
    near = [int(np.argmin(np.abs(mzs - m))) for m in WORLD.contaminant_mz]
    steady = max(near, key=lambda i: (float(np.median(ys[i])), -i))   # the tallest steady background ion
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    for ax, i, title in ((axes[0], pick, "a planted ion"), (axes[1], steady, "a steady background ion")):
        ax.plot(run.scan_rt, ys[i], color="0.3", lw=1)
        for k, colour, label in ((detect_peaks(ys[i], SMOOTH_WINDOW, SNR, noise, BASE_SCANS, PROM_FRAC), "tab:green", "kept"),
                                 (detect_peaks(ys[i], SMOOTH_WINDOW, SNR, noise, BASE_SCANS, 0.0), "tab:red", "S/N rule only")):
            ax.plot(run.scan_rt[k], ys[i][k], "o" if colour == "tab:green" else "x", color=colour, label=label)
        ax.set_title(f"{title}, m/z {mzs[i]:.4f}")
        ax.set_xlabel("retention time (min)")
        ax.set_ylabel("summed intensity")
        ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    _show(fig)
    kept = len(detect_peaks(ys[steady], SMOOTH_WINDOW, SNR, noise, BASE_SCANS, PROM_FRAC))
    loose = len(detect_peaks(ys[steady], SMOOTH_WINDOW, SNR, noise, BASE_SCANS, 0.0))
    print(f"the steady background ion gives {loose} peaks with the signal-to-noise rule alone and {kept} with the prominence rule")


_demo("peak figure", _peaks_figure, needs=_UPTO[2])

## 5. Exercise 3 — `match_features()`, and Exercise 4 — `warp_rt()`

One run is one measurement. To compare runs, or to trust a feature, you need to know which feature
in run B is the same compound's ion as a feature in run A. Two features are candidates for
each other when their m/z agree to within a tolerance in ppm **and** their retention times agree to
within a tolerance in minutes. That is easy when the clocks agree, and they do not.

The standard way out has two steps, and you write one function for each.

- **Match** the features of a run to the reference run with a *wide* retention-time window, so
  that features that drifted still pair up. Exercise 3: take every candidate pair `(i, j)` with
  `|mz_a[i] - mz_b[j]| <= ppm * 1e-6 * mz_b[j]` and `|rt_a[i] - rt_b[j]| <= rt_tol`, order them by
  `|rt_a[i] - rt_b[j]|` and then by `i` and then by `j`, and accept a pair when neither feature has been
  taken yet. Return the accepted index pairs sorted by `i`. The same function does the final
  cross-run matching later, with a narrow window.
- **Warp.** Pairs give points `(retention time in this run, retention time in the reference)`. Draw
  a piecewise-linear curve through them and use it to move every retention time of the run onto
  the reference axis. Exercise 4: between the knots, interpolate linearly; before the first knot
  and after the last, keep the offset of the nearest end knot, so that `rt` goes to
  `rt + (ref_knot - run_knot)` there.

<details><summary>💡 Hint 1 — what to think about</summary>

For matching, think about what goes wrong with the greedy rule if you process features in the order
they are listed rather than the order of how close the pair is: a poor early pair can steal a feature
from a much better later one. So the order is a property of the *pairs*. For warping, what should
happen to a retention time outside the knots, and why is a constant offset a safer guess than
a straight line carried on beyond the last two knots?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Matching: compare every feature of one list with every feature of the other using array
arithmetic, keep the pairs that pass both windows, sort them by the three keys in the order given,
then walk the sorted list once with two "already taken" sets. Warping: an interpolation routine
handles the inside; overwrite the two outside ranges with the end offsets. Matching raises
`ValueError` for lists of different lengths, a `ppm` that is not above 0 or a negative `rt_tol`; warping raises it
for knots that are not two equal-length one-dimensional arrays of at least two points, or
`knots_run` that does not strictly increase.
</details>

In [ ]:
def match_features(mz_a: np.ndarray, rt_a: np.ndarray, mz_b: np.ndarray, rt_b: np.ndarray,
                   ppm: float, rt_tol: float) -> tuple[np.ndarray, np.ndarray]:
    """One-to-one matching of two feature lists by m/z (ppm) and retention time (minutes), closest first.

    Candidate pairs `(i, j)` have `|mz_a[i] - mz_b[j]| <= ppm * 1e-6 * mz_b[j]` and
    `|rt_a[i] - rt_b[j]| <= rt_tol` (the tolerance included). Walk the candidates ordered by
    `|rt_a[i] - rt_b[j]|`, then `i`, then `j`; accept a pair when neither `i` nor `j` has been
    accepted already. Raise `ValueError` if a list's m/z and retention times differ in length, or
    `ppm <= 0`, or `rt_tol < 0`.

    Example:
        >>> ia, ib = match_features(np.array([100.0, 200.0]), np.array([1.0, 2.0]),
        ...                         np.array([200.0001, 100.0]), np.array([2.05, 1.5]), 5.0, 0.1)
        >>> ia.tolist(), ib.tolist()
        ([1], [0])

    Returns:
        `(ia, ib)`: integer arrays of matched indices into list a and list b, sorted by `ia`
        (empty arrays when nothing matches).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_match() -> None:
    ia, ib = match_features(np.array([100.0, 200.0]), np.array([1.0, 2.0]),
                            np.array([200.0001, 100.0]), np.array([2.05, 1.5]), 5.0, 0.1)
    assert (ia.tolist(), ib.tolist()) == ([1], [0]), (
        f"expected ([1], [0]), got ({ia.tolist()}, {ib.tolist()}). Feature 0 of a has the m/z of b[1] but is "
        "0.5 min away, outside a window of 0.1")
    ia, ib = match_features(np.array([100.0, 100.0]), np.array([1.0, 1.375]), np.array([100.0]), np.array([1.25]),
                            5.0, 0.5)
    assert (ia.tolist(), ib.tolist()) == ([1], [0]), (
        f"two features want the same partner; the CLOSER pair (0.125 min) must win, not the first listed: "
        f"expected ([1], [0]), got ({ia.tolist()}, {ib.tolist()})")
    ia, ib = match_features(np.array([100.0, 100.0]), np.array([1.0, 1.25]), np.array([100.0]), np.array([1.125]),
                            5.0, 0.5)
    assert (ia.tolist(), ib.tolist()) == ([0], [0]), (
        f"two pairs equally close (0.125 min): the lower index of list a wins; got ({ia.tolist()}, {ib.tolist()})")
    ia, ib = match_features(np.array([100.0]), np.array([1.0]), np.array([100.0]), np.array([1.25]), 5.0, 0.25)
    assert (ia.tolist(), ib.tolist()) == ([0], [0]), "a retention-time gap exactly equal to rt_tol is a match (<=)"
    ia, ib = match_features(np.array([]), np.array([]), np.array([100.0]), np.array([1.0]), 5.0, 0.1)
    assert len(ia) == 0 and len(ib) == 0, "an empty list matches nothing"
    for args in ((np.array([100.0]), np.array([1.0, 2.0]), np.array([100.0]), np.array([1.0]), 5.0, 0.1),
                 (np.array([100.0]), np.array([1.0]), np.array([100.0]), np.array([1.0]), 0.0, 0.1),
                 (np.array([100.0]), np.array([1.0]), np.array([100.0]), np.array([1.0]), 5.0, -0.1)):
        try:
            match_features(*args)
        except ValueError:
            pass
        else:
            raise AssertionError("mismatched lengths, ppm <= 0 and a negative rt_tol must raise ValueError")
    print("exercise 3 looks right — one-to-one matching, closest pair first")

In [ ]:
_try("exercise 3", _check_match)

In [ ]:
def warp_rt(rt: np.ndarray, knots_run: np.ndarray, knots_ref: np.ndarray) -> np.ndarray:
    """Move retention times onto the reference axis with a piecewise-linear warp through the knots.

    Inside `[knots_run[0], knots_run[-1]]` interpolate linearly between knots. Outside, keep the
    offset of the nearest end knot: `rt + (knots_ref[0] - knots_run[0])` below the first and
    `rt + (knots_ref[-1] - knots_run[-1])` above the last. Raise `ValueError` unless the knots are two
    one-dimensional arrays of the same length of at least 2 and `knots_run` strictly increases.

    Example:
        >>> warp_rt(np.array([1.0, 2.0, 3.0, 5.0]), np.array([1.0, 3.0]), np.array([1.5, 3.5])).tolist()
        [1.5, 2.5, 3.5, 5.5]

    Returns:
        A float array the shape of `rt`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_warp() -> None:
    got = warp_rt(np.array([1.0, 2.0, 3.0, 5.0]), np.array([1.0, 3.0]), np.array([1.5, 3.5]))
    assert np.allclose(got, [1.5, 2.5, 3.5, 5.5], rtol=0, atol=1e-12), (
        f"expected [1.5, 2.5, 3.5, 5.5], got {np.round(got, 6).tolist()}. Beyond the last knot the offset of "
        "that knot (+0.5) carries on; an interpolation routine alone holds the end VALUE instead")
    got = warp_rt(np.array([0.0, 4.0, 8.0]), np.array([1.0, 3.0, 5.0]), np.array([1.0, 4.0, 5.5]))
    assert np.allclose(got, [0.0, 4.75, 8.5], rtol=0, atol=1e-12), (
        f"expected [0.0, 4.75, 8.5], got {np.round(got, 6).tolist()}. Below the first knot the offset is "
        "ref - run at that knot (0.0 here); between knots 2 and 3, 4.0 sits a quarter of the way along")
    got = warp_rt(np.array([[1.0, 2.0], [3.0, 4.0]]), np.array([0.0, 10.0]), np.array([1.0, 11.0]))
    assert got.shape == (2, 2) and np.allclose(got, [[2.0, 3.0], [4.0, 5.0]], rtol=0, atol=1e-12), (
        "the answer has the shape of rt")
    for bad_run, bad_ref in ((np.array([1.0, 1.0]), np.array([1.0, 2.0])), (np.array([2.0, 1.0]), np.array([1.0, 2.0])),
                             (np.array([1.0]), np.array([1.0])), (np.array([1.0, 2.0]), np.array([1.0, 2.0, 3.0]))):
        try:
            warp_rt(np.array([1.5]), bad_run, bad_ref)
        except ValueError:
            pass
        else:
            raise AssertionError(f"knots {bad_run.tolist()}, {bad_ref.tolist()} must raise ValueError")
    print("exercise 4 looks right — retention times warped onto the reference axis")

In [ ]:
_try("exercise 4", _check_warp)

The two functions need a little given glue. `align_run` matches the tallest peaks of a run to
the reference with the wide window, groups the matched pairs into `N_KNOTS` bins by retention time,
and takes the median of each bin, so that one wrong pair cannot bend the curve. `consensus_table`
warps each run, matches it to what has been seen so far with the narrow window, and keeps a feature
only when at least `MIN_RUNS` runs found it. The cells after the glue compare the warp with the
answer key, and then the feature table with and without the warp.

In [ ]:
def align_run(ref, run, ppm: float = MATCH_PPM, anchor_tol: float = ANCHOR_RT_TOL, n_knots: int = N_KNOTS,
              top: int = 150):
    """Given. Knots `(knots_run, knots_ref)` that warp `run` onto `ref`; both are `(mz, rt, height)` tables."""
    pick = np.argsort(-run[2], kind="stable")[:top]
    ia, ib = match_features(run[0][pick], run[1][pick], ref[0], ref[1], ppm, anchor_tol)
    x_run, x_ref = run[1][pick][ia], ref[1][ib]
    order = np.argsort(x_run, kind="stable")
    bins = [b for b in np.array_split(np.arange(len(x_run)), n_knots) if len(b)]
    return (np.array([np.median(x_run[order][b]) for b in bins]),
            np.array([np.median(x_ref[order][b]) for b in bins]))


class Consensus(NamedTuple):
    mz: np.ndarray       # mean m/z over the runs that found the feature
    rt: np.ndarray       # mean retention time on the reference axis
    height: np.ndarray   # mean apex height
    n_runs: np.ndarray   # how many runs found it


def consensus_table(tables, warps, ppm: float = MATCH_PPM, rt_tol: float = MATCH_RT_TOL,
                    min_runs: int = MIN_RUNS) -> Consensus:
    """Given. Merge the runs' peak tables into one feature table, using your `match_features` and `warp_rt`.

    `warps[r]` is `(knots_run, knots_ref)` for run r, or None to leave its retention times alone.
    """
    mzs = [[m] for m in tables[0][0]]
    rts = [[t] for t in tables[0][1]]
    hs = [[h] for h in tables[0][2]]
    for r in range(1, len(tables)):
        m, t, h = tables[r]
        if warps[r] is not None:
            t = warp_rt(t, *warps[r])
        ia, ib = match_features(m, t, np.array([np.mean(v) for v in mzs]), np.array([np.mean(v) for v in rts]),
                                ppm, rt_tol)
        for i, j in zip(ia.tolist(), ib.tolist()):
            mzs[j].append(m[i]); rts[j].append(t[i]); hs[j].append(h[i])
        matched = set(ia.tolist())
        for i in range(len(m)):
            if i not in matched:
                mzs.append([m[i]]); rts.append([t[i]]); hs.append([h[i]])
    n = np.array([len(v) for v in hs])
    keep = n >= min_runs
    return Consensus(np.array([np.mean(v) for v in mzs])[keep], np.array([np.mean(v) for v in rts])[keep],
                     np.array([np.mean(v) for v in hs])[keep], n[keep])


def _tables(w: World, snr: float = SNR, prom_frac: float = PROM_FRAC):
    """Peak tables of every run (cached while your functions are the ones that made them)."""
    return _memo(("tables", w.seed, len(w.runs), snr, prom_frac), ("cluster_mz", "detect_peaks"),
                 lambda: [run_peak_table(run, snr=snr, prom_frac=prom_frac) for run in w.runs])


def _warps(w: World, tables):
    return [None] + [align_run(tables[0], tables[r]) for r in range(1, len(tables))]


def _align_demo() -> None:
    tabs = _tables(WORLD)
    warps = _warps(WORLD, tabs)
    t_grid = np.linspace(2.0, RT_MAX - 2.0, 200)
    print(f"{'run':>4} {'pairs':>6} {'median |dt| before':>20} {'after':>8}    worst error of the warp against the true drift")
    for r in range(1, len(tabs)):
        ia, ib = match_features(tabs[r][0], tabs[r][1], tabs[0][0], tabs[0][1], MATCH_PPM, ANCHOR_RT_TOL)
        before = np.abs(tabs[r][1][ia] - tabs[0][1][ib])
        after = np.abs(warp_rt(tabs[r][1][ia], *warps[r]) - tabs[0][1][ib])
        err = np.abs(warp_rt(_drift(WORLD.drift, r, t_grid), *warps[r]) - t_grid)
        print(f"{r:4d} {len(ia):6d} {np.median(before):16.3f} min {np.median(after):5.3f} min    {err.max():.3f} min "
              f"(from {t_grid[0]:g} to {t_grid[-1]:g} min)")


_demo("alignment", _align_demo, needs=_UPTO[4])

In [ ]:
def _consensus_scores(w: World, c: Consensus) -> tuple[int, float, float]:
    """(features, precision, recall) of a feature table against the planted ions."""
    tags = tag_truth(w, c.mz, c.rt)
    seen = set(np.nonzero(w.ion_height >= DETECTABLE)[0].tolist())
    return (len(tags), float((tags >= 0).mean()) if len(tags) else float("nan"),
            len(set(tags[tags >= 0].tolist()) & seen) / len(seen))


def _consensus_demo() -> None:
    tabs = _tables(WORLD)
    warps = _warps(WORLD, tabs)
    print(f"{'warp':>9} {'rt window':>10} {'min runs':>9} {'features':>9} {'precision':>10} {'recall':>7}")
    for name, use, tol, runs in (("yes", True, MATCH_RT_TOL, MIN_RUNS), ("no", False, MATCH_RT_TOL, MIN_RUNS),
                                 ("no", False, 0.3, MIN_RUNS), ("no", False, 0.6, MIN_RUNS),
                                 ("yes", True, MATCH_RT_TOL, 1)):
        c = consensus_table(tabs, warps if use else [None] * len(tabs), rt_tol=tol, min_runs=runs)
        n, prec, rec = _consensus_scores(WORLD, c)
        print(f"{name:>9} {tol:7g} min {runs:9d} {n:9d} {100 * prec:9.1f}% {100 * rec:6.1f}%")
    loose = _tables(WORLD, snr=1.0)
    lw = _warps(WORLD, loose)
    print("the same, with a low signal-to-noise bar (more bumps get through the peak picking):")
    for runs in (1, MIN_RUNS):
        c = consensus_table(loose, lw, min_runs=runs)
        n, prec, rec = _consensus_scores(WORLD, c)
        print(f"{'yes':>9} {MATCH_RT_TOL:7g} min {runs:9d} {n:9d} {100 * prec:9.1f}% {100 * rec:6.1f}%")


_demo("consensus table", _consensus_demo, needs=_UPTO[4])

Read the first two rows against each other: they differ only in the warp, and nothing else. Then
read down the unwarped rows as the window widens, and compare each with the first row: what a
wider window does to recall and to precision is on the page, and it need not be a gain. How much
the warp matters depends on how far the runs drifted, which section 2 printed: in a world whose
runs drift less than a peak is wide, the rows come closer together. The last rows are the other
filter. A bump that is not a compound is unlikely to turn up in the same place in a
second run, so asking for two runs removes it, at the price of any compound that happened to be
found in only one. Which price to pay is a decision about the study, and the table is the evidence for it.

## 6. Exercise 5 — `group_isotopes()`

The consensus table now lists ions, one per m/z and retention time. A compound with one adduct
already contributes up to three of them: its monoisotopic peak and the two above it. Isotope peaks
can be recognised by a rule: they are exactly one carbon-13-minus-carbon-12 mass difference above
the peak before, they co-elute with it, and (for molecules as small as these) none is taller than
the one before it; `max_ratio` is the knob for that last condition. This world is simpler than
chemistry: its isotope peaks come from carbon-13 alone. In a real molecule nitrogen-15, deuterium,
oxygen and sulfur isotopes, and potassium-41 in a potassium adduct, add peaks of their own, some a
few millidaltons from the carbon-13 step, so this rule and the M+1 ratio of exercise 8 are a
first-order model.

**The rule.** Feature `j` follows feature `i` as an isotope peak when all three hold:

- `|(mz[j] - mz[i]) - ISO_DELTA| <= ppm * 1e-6 * mz[i]` (one carbon-13 step; the ions have charge 1);
- `|rt[j] - rt[i]| <= rt_tol` (co-elution);
- `height[j] <= max_ratio * height[i]` (an isotope peak is no taller than the peak before it).

Join every such pair, and join the pairs end to end: M+1 follows M, M+2 follows M+1, so all three
end up in one group. Return one label per feature. The given `connected_labels(n, pairs)` does
the joining and gives each group the smallest index in it as its label, which makes the answer
independent of the order of the features, so what you write is only the *pair finding*.

<details><summary>💡 Hint 1 — what to think about</summary>

Take one feature at a time and ask which other features follow it: all three tests are array
comparisons against the whole list at once. Which way does the mass test run, and what does that say
about a pair where the heavier peak is the taller one? The grouping is a graph question, so you
never need to decide who is "the" monoisotopic peak yourself.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Loop over the features. For each, build one boolean array over all features from the three
conditions (the mass window measured against this feature's m/z), turn the true positions into
index pairs with this feature, and collect them. Hand all pairs to `connected_labels`. Reject
arrays that are not one-dimensional and of one length, a `ppm` or `max_ratio` that is not above
0 and a negative `rt_tol` with `ValueError`.
</details>

In [ ]:
def group_isotopes(mz: np.ndarray, rt: np.ndarray, height: np.ndarray, ppm: float = 5.0,
                   rt_tol: float = 0.05, max_ratio: float = 1.0) -> np.ndarray:
    """Group features that are isotope peaks of each other; label = smallest index in the group.

    `j` follows `i` when `|(mz[j] - mz[i]) - ISO_DELTA| <= ppm * 1e-6 * mz[i]`,
    `|rt[j] - rt[i]| <= rt_tol` and `height[j] <= max_ratio * height[i]`. Pairs are joined end to
    end (use `connected_labels`). Raise `ValueError` for arrays that are not one-dimensional and of one
    length, `ppm <= 0`, `max_ratio <= 0` or `rt_tol < 0`.

    Example:
        >>> mz = np.array([300.0, 301.00335, 302.0067, 500.0])
        >>> group_isotopes(mz, np.array([5.0, 5.01, 5.0, 5.0]), np.array([1000.0, 200.0, 20.0, 900.0])).tolist()
        [0, 0, 0, 3]

    Returns:
        An integer array of labels, one per feature.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_isotopes() -> None:
    mz = np.array([300.0, 301.00335, 302.0067, 500.0])
    got = group_isotopes(mz, np.array([5.0, 5.01, 5.0, 5.0]), np.array([1000.0, 200.0, 20.0, 900.0]))
    assert got.tolist() == [0, 0, 0, 3], (
        f"expected [0, 0, 0, 3], got {got.tolist()}. M+2 follows M+1, which follows M: join the pairs END TO END")
    got = group_isotopes(np.array([300.0, 301.00335]), np.array([5.0, 5.0]), np.array([100.0, 200.0]))
    assert got.tolist() == [0, 1], (
        f"a heavier peak TALLER than the lighter one is not its isotope peak: expected [0, 1], got {got.tolist()}")
    got = group_isotopes(np.array([300.0, 301.00335]), np.array([5.0, 6.0]), np.array([1000.0, 100.0]))
    assert got.tolist() == [0, 1], f"peaks a minute apart do not co-elute: expected [0, 1], got {got.tolist()}"
    got = group_isotopes(np.array([301.00335, 300.0]), np.array([5.0, 5.0]), np.array([100.0, 1000.0]))
    assert got.tolist() == [0, 0], (
        f"the answer must not depend on the order of the features: expected [0, 0], got {got.tolist()}")
    got = group_isotopes(np.array([300.0, 301.0]), np.array([5.0, 5.0]), np.array([1000.0, 100.0]))
    assert got.tolist() == [0, 1], (
        f"a step of 1.0000 (not the carbon-13 step) is outside the mass window: expected [0, 1], got {got.tolist()}")
    assert len(group_isotopes(np.array([]), np.array([]), np.array([]))) == 0, "no features, no labels"
    print("exercise 5 looks right — isotope peaks joined to the peak below them")

In [ ]:
_try("exercise 5", _check_isotopes)

## 7. Exercise 6 — `group_adducts()`

After isotopes are folded in, a compound can still show up as several features, one per adduct. Two
ions of the *same neutral molecule* have m/z values that differ by the difference between their two
adduct shifts, and they elute together. So the test for "same molecule" is the same one, applied to
neutral masses: feature `i` read as adduct `a` has neutral mass `mz[i] - ADDUCT_SHIFT[a]`.

**The rule.** Features `i` and `j` are linked when `|rt[i] - rt[j]| <= rt_tol` and there are two
*different* adducts `a` and `b` (indices into `ADDUCTS`) for which
`|(mz[j] - ADDUCT_SHIFT[b]) - (mz[i] - ADDUCT_SHIFT[a])| <= ppm * 1e-6 * (mz[i] - ADDUCT_SHIFT[a])`.
Join the links end to end with `connected_labels`, as before. Pass in only the **monoisotopic**
features (the notebook does): each isotope series has already been folded into its first peak.

<details><summary>💡 Hint 1 — what to think about</summary>

For one feature, which pairs of adducts are there, and what does each pair say about where the
other feature must be? Two adducts must be different: a feature compared with itself, or with a
second feature at the same m/z, under the *same* adduct, is not evidence of an adduct pair.
What condition on retention time keeps two compounds that merely have related masses apart?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Loop over features. For each one, build the boolean array of features that co-elute with it. For
each ordered pair of different adducts, compute the neutral mass this feature would have as the
first adduct, and test the whole list against it under the second adduct, within the ppm window
of that neutral mass. Turn the matching positions (leaving out the feature itself) into index pairs, and give
all of them to `connected_labels`. Argument checks as for the isotope exercise, without `max_ratio`.
</details>

In [ ]:
def group_adducts(mz: np.ndarray, rt: np.ndarray, ppm: float = 5.0, rt_tol: float = 0.05) -> np.ndarray:
    """Group features that are different adducts of one neutral molecule; label = smallest index.

    Features `i`, `j` are linked when `|rt[i] - rt[j]| <= rt_tol` and, for two different adducts
    `a != b`, `|(mz[j] - ADDUCT_SHIFT[b]) - (mz[i] - ADDUCT_SHIFT[a])| <= ppm * 1e-6 * (mz[i] - ADDUCT_SHIFT[a])`.
    Links are joined end to end (use `connected_labels`). Raise `ValueError` for arrays that are not
    one-dimensional and of one length, `ppm <= 0` or `rt_tol < 0`.

    Example:
        >>> mz = np.array([301.0073, 322.9892, 410.0])
        >>> group_adducts(mz, np.array([5.0, 5.02, 5.0])).tolist()
        [0, 0, 2]

    Returns:
        An integer array of labels, one per feature.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_adducts() -> None:
    mz = np.array([301.0073, 322.9892, 410.0])
    got = group_adducts(mz, np.array([5.0, 5.02, 5.0]))
    assert got.tolist() == [0, 0, 2], (
        f"expected [0, 0, 2], got {got.tolist()}. The first two are [M+H]+ and [M+Na]+ of a molecule of mass "
        "about 300.0, eluting 0.02 min apart; the third is unrelated")
    got = group_adducts(mz, np.array([5.0, 7.0, 5.0]))
    assert got.tolist() == [0, 1, 2], (
        f"the same two masses two minutes apart are two compounds: expected [0, 1, 2], got {got.tolist()}. "
        "The co-elution test is missing")
    got = group_adducts(np.array([322.9892, 410.0, 301.0073]), np.array([5.0, 5.0, 5.0]))
    assert got.tolist() == [0, 1, 0], (
        f"the answer must not depend on the order of the features: expected [0, 1, 0], got {got.tolist()}")
    chain = group_adducts(np.array([301.0073, 322.9892, 338.9631]), np.array([5.0, 5.0, 5.0]))
    assert chain.tolist() == [0, 0, 0], (
        f"[M+H]+, [M+Na]+ and [M+K]+ of one molecule form one group: expected [0, 0, 0], got {chain.tolist()}")
    same = group_adducts(np.array([301.0073, 301.0073]), np.array([5.0, 5.0]))
    assert same.tolist() == [0, 1], (
        f"two features at the same m/z are not two adducts of one molecule: expected [0, 1], got {same.tolist()}")
    assert len(group_adducts(np.array([]), np.array([]))) == 0, "no features, no labels"
    print("exercise 6 looks right — adduct ions of one molecule joined by co-elution and mass")

In [ ]:
_try("exercise 6", _check_adducts)

## 8. Exercise 7 — `neutral_mass()`

A group of adduct ions has a neutral mass, and an annotation searches the library for that mass,
not for the m/z of any one ion. To find it you must decide which adduct each ion is, and you
usually cannot be sure. Use the evidence the group offers:

1. Take the **tallest** ion of the group (the lowest index among equal heights). Suppose it is
   adduct `a`; its neutral mass would be `mz[top] - ADDUCT_SHIFT[a]`.
2. For each of the four suppositions, count the ions of the group that the supposition **explains**:
   an ion `x` is explained when `|(x - ADDUCT_SHIFT[b]) - M| <= ppm * 1e-6 * M` for *some* adduct `b`.
3. Pick the supposition that explains the most ions; ties go to the earlier adduct in `ADDUCTS`
   (which puts `[M+H]+` first, the usual guess for an ion with no evidence).

Return the neutral mass and the number of ions explained. A group of one ion has no evidence,
so it is read as `[M+H]+`: that is the assumption that goes wrong for a molecule that shows only a sodium
adduct, and the lesson measures how often.

<details><summary>💡 Hint 1 — what to think about</summary>

The four suppositions are all about the *same* ion, the tallest, so there are only four neutral masses
to try. How would you count how many members of the group a given neutral mass explains, if each
member may be any of the four adducts? The tie rule is what makes a single ion come out as `[M+H]+`.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Find the position of the tallest ion with `argmax`, which already returns the first of equal values.
Loop over the adducts in table order. For each, compute the candidate mass; for each member, test
whether subtracting any adduct shift lands within the ppm window of that candidate; count. Keep a
candidate only when it beats the best count so far (strictly), which gives the tie rule for free.
Return a plain float and a plain int. `ValueError` for empty or mismatched arrays and a `ppm` not above 0.
</details>

In [ ]:
def neutral_mass(mz: np.ndarray, height: np.ndarray, ppm: float = 5.0) -> tuple[float, int]:
    """Neutral mass of a group of adduct ions, and how many of its ions that reading explains.

    The candidate masses are `mz[top] - ADDUCT_SHIFT[a]` for the tallest ion `top` (lowest index
    on a tie) and each adduct `a`, in the order of `ADDUCTS`. A candidate `M` explains an ion `x`
    when `|(x - ADDUCT_SHIFT[b]) - M| <= ppm * 1e-6 * M` for some adduct `b`. The candidate that
    explains the most ions wins; ties go to the earlier adduct. Raise `ValueError` if the arrays are
    empty, not one-dimensional or of different lengths, or `ppm <= 0`.

    Example:
        >>> M, n = neutral_mass(np.array([301.0073, 322.9892]), np.array([800.0, 1500.0]))
        >>> round(M, 3), n
        (300.0, 2)

    Returns:
        `(M, n_explained)`: a float and an int.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_neutral() -> None:
    M, n = neutral_mass(np.array([301.0073, 322.9892]), np.array([800.0, 1500.0]))
    assert (round(M, 3), n) == (300.0, 2), (
        f"expected (300.0, 2), got ({round(M, 3)}, {n}). The tallest ion is the sodium adduct; reading it as "
        "[M+H]+ explains only itself, reading it as [M+Na]+ explains both")
    M, n = neutral_mass(np.array([322.9892]), np.array([100.0]))
    assert (round(M, 3), n) == (321.982, 1), (
        f"a lone ion has no evidence and is read as [M+H]+: expected (321.982, 1), got ({round(M, 3)}, {n})")
    M, n = neutral_mass(np.array([301.0073, 322.9892]), np.array([900.0, 900.0]))
    assert (round(M, 3), n) == (300.0, 2), (
        f"equal heights: the lowest index is the tallest; expected (300.0, 2), got ({round(M, 3)}, {n})")
    M, n = neutral_mass(np.array([301.0073, 500.0]), np.array([900.0, 100.0]))
    assert (round(M, 3), n) == (300.0, 1), (
        f"an ion that fits no reading is simply not explained: expected (300.0, 1), got ({round(M, 3)}, {n})")
    assert isinstance(M, float) and isinstance(n, int), "return a plain float and a plain int"
    for bad in ((np.array([]), np.array([])), (np.array([301.0]), np.array([1.0, 2.0]))):
        try:
            neutral_mass(*bad)
        except ValueError:
            pass
        else:
            raise AssertionError("empty or mismatched arrays must raise ValueError")
    print("exercise 7 looks right — a neutral mass for a group, with the evidence counted")

In [ ]:
_try("exercise 7", _check_neutral)

The given `assemble_groups` strings exercises 5 to 7 together: fold isotope series into their
monoisotopic features, group those by adduct, give each group a neutral mass, a retention time, and
the ratio of its tallest ion's M+1 peak to that ion, which the annotation will use. The answer key
lets us ask how the grouping did.

In [ ]:
class Groups(NamedTuple):
    M: np.ndarray            # neutral mass of each group
    rt: np.ndarray          # mean retention time of its monoisotopic ions
    r1: np.ndarray          # M+1 / M intensity ratio of its tallest ion, NaN if no M+1 peak was found
    members: tuple          # per group, the indices of ALL its consensus features (isotope peaks too)
    n_mono: np.ndarray      # how many monoisotopic ions the group has
    n_explained: np.ndarray # how many of them the neutral mass explains


def assemble_groups(c: Consensus, ppm: float = GROUP_PPM, rt_tol: float = GROUP_RT_TOL) -> Groups:
    """Given. Isotope grouping, adduct grouping and a neutral mass for each group, using exercises 5 to 7."""
    iso = group_isotopes(c.mz, c.rt, c.height, ppm, rt_tol)
    roots = np.nonzero(iso == np.arange(len(iso)))[0]
    adduct = group_adducts(c.mz[roots], c.rt[roots], ppm, rt_tol)
    Ms, rts, r1s, members, n_mono, n_expl = [], [], [], [], [], []
    for g in np.unique(adduct):
        mono = roots[adduct == g]
        M, explained = neutral_mass(c.mz[mono], c.height[mono], ppm)
        top = mono[int(np.argmax(c.height[mono]))]
        series = np.nonzero(iso == iso[top])[0]
        plus1 = series[np.abs((c.mz[series] - c.mz[top]) - ISO_DELTA) <= ppm * 1e-6 * c.mz[top]]
        r1s.append(float(c.height[plus1[0]] / c.height[top]) if len(plus1) else float("nan"))
        Ms.append(M); rts.append(float(c.rt[mono].mean())); n_mono.append(len(mono)); n_expl.append(explained)
        members.append(np.nonzero(np.isin(iso, iso[mono]))[0])
    return Groups(np.array(Ms), np.array(rts), np.array(r1s), tuple(members), np.array(n_mono), np.array(n_expl))


def group_truth(w: World, c: Consensus, g: Groups) -> np.ndarray:
    """Given (answer key). The library index of the planted compound each group is, or -1 if it is none.

    A group is the compound that most of its features are tagged as. Groups that mix compounds are
    counted separately by `_group_purity`.
    """
    tags = tag_truth(w, c.mz, c.rt)
    comp = np.where(tags >= 0, w.ion_comp[np.maximum(tags, 0)], -1)
    out = []
    for mem in g.members:
        got = comp[mem][comp[mem] >= 0]
        out.append(int(np.bincount(got).argmax()) if len(got) else -1)
    return np.array(out, dtype=int)


def _group_purity(w: World, c: Consensus, g: Groups) -> tuple[int, int, int]:
    """(groups, groups mixing different compounds or a compound with a non-compound, compounds split over groups)."""
    tags = tag_truth(w, c.mz, c.rt)
    comp = np.where(tags >= 0, w.ion_comp[np.maximum(tags, 0)], -1)
    mixed = sum(len(set(comp[mem].tolist())) > 1 for mem in g.members)
    homes: dict[int, set] = {}
    for k, mem in enumerate(g.members):
        for q in set(comp[mem].tolist()) - {-1}:
            homes.setdefault(q, set()).add(k)
    return len(g.members), mixed, sum(len(v) > 1 for v in homes.values())


def _grouping_demo() -> None:
    tabs = _tables(WORLD)
    c = consensus_table(tabs, _warps(WORLD, tabs))
    iso = group_isotopes(c.mz, c.rt, c.height, GROUP_PPM, GROUP_RT_TOL)
    roots = np.nonzero(iso == np.arange(len(iso)))[0]
    adduct = group_adducts(c.mz[roots], c.rt[roots], GROUP_PPM, GROUP_RT_TOL)
    seen = set(WORLD.ion_comp[WORLD.ion_height >= DETECTABLE].tolist())
    print(f"consensus features (ions): {len(c.mz)}")
    print(f"after folding isotope peaks into their monoisotopic peak: {len(roots)}")
    print(f"after joining the adduct ions of one molecule: {len(np.unique(adduct))}")
    print(f"planted compounds with at least one ion tall enough to find: {len(seen)}; "
          f"planted compounds in the sample in all: {len(WORLD.planted)}")
    print()
    print(f"{'co-elution window':>18} {'groups':>7} {'mixed groups':>13} {'compounds split':>16}")
    for tol in (0.02, GROUP_RT_TOL, 0.5, 1000.0):
        g = assemble_groups(c, GROUP_PPM, tol)
        n, mixed, split = _group_purity(WORLD, c, g)
        print(f"{tol:13g} min {n:7d} {mixed:13d} {split:16d}")


_demo("grouping", _grouping_demo, needs=_UPTO[7])

Read the table from top to bottom. The window is the only thing that changes. A window narrower
than the retention-time scatter of one compound's ions splits it; a window that ignores
co-elution joins ions that only have related masses, which are the "mixed groups" (the traps of
section 2 are made to be joined this way). Which rows are the ones an analyst would work in is
for the table to say.

In [ ]:
def _neutral_mass_demo() -> None:
    tabs = _tables(WORLD)
    c = consensus_table(tabs, _warps(WORLD, tabs))
    g = assemble_groups(c)
    truth = group_truth(WORLD, c, g)
    tags = tag_truth(WORLD, c.mz, c.rt)
    lib = WORLD.library
    ok = np.array([t >= 0 and abs(g.M[k] - lib.mass[t]) <= ANNOT_PPM * 1e-6 * lib.mass[t]
                   for k, t in enumerate(truth)])
    print(f"{'ions in the group':>20} {'groups':>7} {'neutral mass right':>19}")
    for label, sel in (("one", g.n_mono == 1), ("two or more", g.n_mono >= 2)):
        print(f"{label:>20} {int(sel.sum()):7d} {int(ok[sel].sum()):19d}")
    single = np.nonzero((g.n_mono == 1) & (truth >= 0))[0]
    kind = []
    for k in single:
        mono = [i for i in g.members[k] if tags[i] >= 0 and WORLD.ion_iso[tags[i]] == 0]
        kind.append(WORLD.ion_adduct[tags[mono[0]]] if mono else -1)
    kind = np.array(kind)
    print("a group of one ion, by the adduct that ion really is (a lone ion is read as [M+H]+):")
    for a in range(len(ADDUCTS)):
        sel = kind == a
        if sel.any():
            print(f"{ADDUCT_NAMES[a]:>20} {int(sel.sum()):7d} {int(ok[single][sel].sum()):19d}")


_demo("neutral masses", _neutral_mass_demo, needs=_UPTO[7])

Two ions are evidence about the adduct; one ion is a guess. When the guess is [M+H]+ and the ion
is really [M+Na]+, the neutral mass is out by the difference between the two adducts, and that
is the mass of a different compound if the library has one there. Section 9 searches the library
with these masses, so the errors made here are about to become wrong names.

## 9. Exercise 8 — `annotate()`

The Metabolomics Standards Initiative asks authors to report the **level of identification** of
every metabolite, and its 2007 paper names four (`claims.yaml`; source read on 2 October 2026, and as
of that date this is what it says). **Level 1**, identified compounds, needs at least two
independent and orthogonal data compared with an authentic standard run under identical
conditions. **Level 2**, putatively annotated compounds, is without a standard, on
physicochemical properties or spectral similarity to a library. **Level 3**, putatively
characterised compound classes. **Level 4**, unknown compounds, which can still be told apart and
quantified. The paper defines the levels; it does not turn evidence into a level in code. The
rule below is this lesson's own illustration of the *form* of the thing; a lab would set its own,
and it is not the initiative's. Its level 1 is weaker than the paper's minimum: only the retention
time is compared with an authentic standard; the accurate mass is compared with the library's
calculated mass, not with the standard run under identical conditions, and other candidates
without a standard may still survive beside the one that is named.

In [ ]:
# The isotope-pattern test below, in numbers: the M+1 / M ratio that a carbon count predicts in this
# world (carbon-13 only), and the band of measured ratios that the annotation's window accepts.
for _n_c in (5, 10, 20, 40):
    _e = _n_c * C13_ABUNDANCE / (1.0 - C13_ABUNDANCE)
    print(f"{_n_c:2d} carbons: expected M+1 / M ≈ {_e:.3f}; accepted from ≈ {_e * (1 - ISO_TOL):.3f} "
          f"to ≈ {_e * (1 + ISO_TOL):.3f}")

**What you have** for a group: its neutral mass `M`, its retention time `rt`, the M+1 / M ratio `r1`
of its tallest ion (NaN if no M+1 peak was found), and the `Library`. **The rule, in this order:**

1. *Candidates:* library entries with `|M - mass| <= ppm * 1e-6 * mass`.
2. *Retention time:* drop a candidate that has a standard (`std_rt` is not NaN) whose `std_rt` differs
   from `rt` by more than `rt_tol`: an authentic standard that elutes elsewhere is not this compound.
3. *Isotope pattern:* if `r1` is not NaN, keep a candidate only when `|r1 - e| <= iso_tol * e`, where
   `e = n_carbon * C13_ABUNDANCE / (1 - C13_ABUNDANCE)`. If `r1` is NaN the pattern says nothing
   and nobody is dropped for it.
4. *Level.* The **survivors** are what is left. **Level 1:** exactly one survivor has a standard,
   and its name is reported. **Level 2:** no survivor has a standard, exactly one survivor is left
   and `r1` was available: its name. **Level 3:** otherwise, if there is at least one survivor and
   all share one class: the class. **Level 4:** otherwise: no name (`None`).

The result also counts the survivors. Notice what the rule refuses to do: a mass that matches
exactly one entry, with no isotope pattern and no standard, is *not* reported as a name. A value
exactly on the edge of the mass window or of the isotope-ratio window is not graded: neither edge
is exact in binary floating point.

<details><summary>💡 Hint 1 — what to think about</summary>

Filter step by step and keep the survivors as you go; the level depends on what is left, not on the
first thing that matched. What distinguishes level 2 from the level 3 that one survivor can also give?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Collect candidates by mass. Remove those with a standard that disagrees in time. If `r1` is a number,
remove those whose expected ratio disagrees. Among the survivors, pick out the ones that have a
standard. Test the four outcomes in order from level 1 down; the first that applies is the answer.
Raise `ValueError` for a `ppm` or `iso_tol` not above 0 or a negative `rt_tol`.
</details>

In [ ]:
class Annotation(NamedTuple):
    level: int           # 1 to 4
    name: Any            # compound name (levels 1 and 2), class (level 3), None (level 4)
    n_candidates: int    # survivors of the filters


def annotate(M: float, rt: float, r1: float, lib: Library, ppm: float = 5.0, rt_tol: float = 0.1,
             iso_tol: float = 0.3) -> Annotation:
    """Report a group at an MSI-style level from its neutral mass, retention time and M+1 ratio.

    Apply, in order: mass candidates (`ppm`), drop candidates whose standard elutes more than `rt_tol`
    away, drop candidates whose expected M+1 / M ratio differs from `r1` by more than `iso_tol` of
    itself (only if `r1` is not NaN). Level 1: exactly one survivor has a standard. Level 2: none
    has one, exactly one survivor is left and `r1` was available. Level 3: otherwise, at least one
    survivor, all of one class. Level 4: otherwise. Raise `ValueError` for `ppm <= 0`, `iso_tol <= 0`
    or `rt_tol < 0`.

    Example:
        >>> lib = Library(("A", "B"), np.array([300.0, 450.0]), np.array([15, 20]), ("acid", "sugar"),
        ...               np.array([5.0, np.nan]))
        >>> annotate(300.0, 5.05, 0.16, lib)
        Annotation(level=1, name='A', n_candidates=1)

    Returns:
        An `Annotation(level, name, n_candidates)`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


_CHECK_LIB = Library(("A", "B", "C", "D"), np.array([300.0, 300.0, 300.0005, 400.0]), np.array([15, 15, 15, 20]),
                     ("acid", "acid", "amine", "sugar"), np.array([5.0, np.nan, np.nan, np.nan]))


def _check_annotate() -> None:
    e15 = 15 * C13_ABUNDANCE / (1 - C13_ABUNDANCE)
    e20 = 20 * C13_ABUNDANCE / (1 - C13_ABUNDANCE)
    lib = _CHECK_LIB
    got = annotate(300.0, 5.05, e15, lib)
    assert got == Annotation(1, "A", 3), (
        f"entry A has a standard that elutes within 0.1 min and matches the mass: expected level 1, 'A', 3 "
        f"survivors; got {tuple(got)}")
    got = annotate(300.0, 6.0, e15, lib)
    assert got == Annotation(4, None, 2), (
        f"A's standard elutes a minute away, so A is dropped; B (acid) and C (amine) survive and differ in class: "
        f"expected level 4, None, 2; got {tuple(got)}")
    got = annotate(400.0, 3.0, e20, lib)
    assert got == Annotation(2, "D", 1), (
        f"one survivor, mass and isotope pattern agree, no standard: expected level 2, 'D'; got {tuple(got)}")
    got = annotate(400.0, 3.0, float("nan"), lib)
    assert got == Annotation(3, "sugar", 1), (
        f"a unique mass match with no isotope pattern is NOT a name: expected level 3, 'sugar'; got {tuple(got)}")
    got = annotate(400.0, 3.0, 0.5, lib)
    assert got == Annotation(4, None, 0), (
        f"an M+1 ratio of 0.5 is far from the 20 carbons of D: expected level 4, None, 0; got {tuple(got)}")
    got = annotate(300.0, 6.0, float("nan"), lib._replace(klass=("acid", "acid", "acid", "sugar")))
    assert got == Annotation(3, "acid", 2), (
        f"B and C survive and share a class: expected level 3, 'acid', 2; got {tuple(got)}")
    got = annotate(350.0, 3.0, e15, lib)
    assert got == Annotation(4, None, 0), f"nothing within the mass window: expected level 4; got {tuple(got)}"
    for bad in ((0.0, 0.1, 0.3), (5.0, -0.1, 0.3), (5.0, 0.1, 0.0)):
        try:
            annotate(300.0, 5.0, e15, lib, *bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"annotate(..., ppm, rt_tol, iso_tol) = {bad} must raise ValueError")
    print("exercise 8 looks right — a level, and the name or class the evidence supports")

In [ ]:
_try("exercise 8", _check_annotate)

## 10. Exercise 9 — `score_annotations()`

An annotation is only as good as the answer key says. Score the groups: for each level, how many
groups were reported at it and how many of those were **wrong**. A group is wrong when it was
reported at level 1 or 2 and its name is not the planted compound's name; when it was reported at
level 3 and its class is not the planted compound's class; and, at levels 1 to 3, whenever the group is
not a planted compound at all (`true_name` is `None`). Level 4 makes no claim, so it can be
counted but never wrong. Return `{1: (n, wrong), 2: ..., 3: ..., 4: ...}` with every level
present, even those with no groups.

<details><summary>💡 Hint 1 — what to think about</summary>

Two lists can both hold `None` at the same position without that being agreement: what does a
name of `None` mean at each level, and what should a true name of `None` do at level 3?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Start with four zero counters pairs. Walk the four lists together; add one to the group count of its level;
decide whether it is wrong by its level's own rule; add one to the wrong count if so. Reject
lists of different lengths and a level outside 1 to 4 with `ValueError`.
</details>

In [ ]:
def score_annotations(levels, names, true_names, true_classes) -> dict:
    """Per level, `(groups reported, groups wrong)`, scored against the planted compounds.

    Level 1 or 2 is wrong unless `true_names` is not None and equals the reported name. Level 3 is
    wrong unless `true_classes` is not None and equals the reported class. Level 4 is never wrong.
    All four levels appear in the result. Raise `ValueError` if the four lists differ in length or a
    level is not 1, 2, 3 or 4.

    Example:
        >>> score_annotations([1, 3, 2, 4], ["A", "acid", "B", None], ["A", "sugar", None, "C"],
        ...                   ["acid", "sugar", None, "amine"])
        {1: (1, 0), 2: (1, 1), 3: (1, 1), 4: (1, 0)}

    Returns:
        A dict mapping each of 1, 2, 3, 4 to a tuple of two ints.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_scoring() -> None:
    got = score_annotations([1, 3, 2, 4], ["A", "acid", "B", None], ["A", "sugar", None, "C"],
                            ["acid", "sugar", None, "amine"])
    assert got == {1: (1, 0), 2: (1, 1), 3: (1, 1), 4: (1, 0)}, (
        f"expected {{1: (1, 0), 2: (1, 1), 3: (1, 1), 4: (1, 0)}}, got {got}. Level 2 with a true name of None is "
        "wrong; level 3 is judged on the CLASS; level 4 is never wrong")
    got = score_annotations([3, 3, 1], ["acid", "acid", "A"], ["x", "y", "A"], ["acid", None, "acid"])
    assert got == {1: (1, 0), 2: (0, 0), 3: (2, 1), 4: (0, 0)}, (
        f"a level-3 group whose true class is None is wrong, and a level with no groups still appears: got {got}")
    got = score_annotations([], [], [], [])
    assert got == {1: (0, 0), 2: (0, 0), 3: (0, 0), 4: (0, 0)}, f"no groups, all zeros: got {got}"
    for bad in (([1], ["A"], ["A"], []), ([5], ["A"], ["A"], ["acid"]), ([0], ["A"], ["A"], ["acid"])):
        try:
            score_annotations(*bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"score_annotations{bad!r} must raise ValueError")
    print("exercise 9 looks right — groups and wrong groups per level")

In [ ]:
_try("exercise 9", _check_scoring)

Now run the whole chain on the world and read the scores. `run_pipeline` (given) uses your nine
functions end to end. Everything printed below comes from the answer key, which an analyst of
real data would not have; the point of the lesson is that you can now see what it would say.

In [ ]:
class Pipeline(NamedTuple):
    consensus: Consensus
    groups: Groups
    truth: np.ndarray        # library index of the planted compound each group is, or -1
    annotations: list
    scores: dict


_DEPS = ("cluster_mz", "detect_peaks", "match_features", "warp_rt", "group_isotopes", "group_adducts",
         "neutral_mass")


def run_pipeline(w: World, lib: Library | None = None, drop_pattern: bool = False) -> Pipeline:
    """Given. Peaks, alignment, consensus, grouping, annotation and scoring for one world.

    `lib` replaces the world's library for the search (the answer key still uses the world's own);
    `drop_pattern=True` hides every M+1 / M ratio, so isotope patterns cannot be used.
    """
    def make():
        tabs = _tables(w)
        c = consensus_table(tabs, _warps(w, tabs))
        g = assemble_groups(c)
        return c, g, group_truth(w, c, g)
    c, g, truth = _memo(("pipeline", w.seed, len(w.runs)), _DEPS, make)
    lib_used = w.library if lib is None else lib
    ann = [annotate(g.M[k], g.rt[k], float("nan") if drop_pattern else g.r1[k], lib_used,
                    ANNOT_PPM, ANNOT_RT_TOL, ISO_TOL) for k in range(len(g.M))]
    names = [None if t < 0 else w.library.names[t] for t in truth]
    classes = [None if t < 0 else w.library.klass[t] for t in truth]
    return Pipeline(c, g, truth, ann, score_annotations([a.level for a in ann], [a.name for a in ann], names, classes))


def _level_table(p: Pipeline) -> None:
    print(f"{'level':>6} {'what was shown':<46} {'groups':>7} {'wrong':>6} {'false-annotation rate':>22}")
    what = {1: "name; mass + a standard's retention time", 2: "name; mass + isotope pattern, no standard",
            3: "class only; mass, ambiguous or no pattern", 4: "nothing claimed"}
    for level in (1, 2, 3, 4):
        n, wrong = p.scores[level]
        rate = f"{100 * wrong / n:20.1f} %" if n and level < 4 else f"{'-':>22}"
        print(f"{level:6d} {what[level]:<46} {n:7d} {wrong:6d} {rate}")


def _annotation_demo() -> None:
    p = run_pipeline(WORLD)
    _level_table(p)
    seen = set(WORLD.ion_comp[WORLD.ion_height >= DETECTABLE].tolist())
    lib = WORLD.library
    right = {1: set(), 2: set(), 3: set()}
    for t, a in zip(p.truth, p.annotations):
        if t >= 0 and a.level in (1, 2) and a.name == lib.names[t]:
            right[a.level].add(int(t))
        if t >= 0 and a.level == 3 and a.name == lib.klass[t]:
            right[3].add(int(t))
    named = right[1] | right[2]
    print(f"\nof the {len(seen)} planted compounds with an ion tall enough to find: {len(seen & right[1])} named at level 1, "
          f"{len(seen & right[2])} at level 2, {len(seen & (right[3] - named))} given only a correct class at level 3, "
          f"{len(seen - named - right[3])} left at level 4 or wrongly annotated")


_demo("annotation levels", _annotation_demo, needs=_UPTO[9])

Each row of that table is a promise: the false-annotation rate is how often the promise was broken.
Two caveats belong next to it. The rates rest on a handful of groups per row, so one group moves
them by a lot (section 11 repeats the run on other worlds). And they are the rates *of this
rule on this world*: a laxer isotope window or a library with more near neighbours would move
them. The last table of this section tests the rule by taking evidence away.

In [ ]:
def _ladder_demo() -> None:
    p = run_pipeline(WORLD)
    lib = WORLD.library
    c, g = p.consensus, p.groups
    tags = tag_truth(WORLD, c.mz, c.rt)
    comp = np.where(tags >= 0, WORLD.ion_comp[np.maximum(tags, 0)], -1)
    iso = group_isotopes(c.mz, c.rt, c.height, GROUP_PPM, GROUP_RT_TOL)
    roots = np.nonzero(iso == np.arange(len(iso)))[0]

    def nearest(M: float) -> int:
        cand = np.nonzero(np.abs(lib.mass - M) <= ANNOT_PPM * 1e-6 * lib.mass)[0]
        return int(cand[np.argmin(np.abs(lib.mass[cand] - M))]) if len(cand) else -1

    named_a = [(nearest(c.mz[i] - ADDUCT_SHIFT[0]), comp[i]) for i in roots]
    named_b = [(nearest(M), t) for M, t in zip(g.M, p.truth)]
    lv12 = [(lib.names.index(a.name), t) for a, t in zip(p.annotations, p.truth) if a.level in (1, 2)]
    print(f"{'how the name was reached':<56} {'named':>6} {'wrong':>6} {'wrong share':>12}")
    for label, pairs in (("each ion alone, read as [M+H]+, nearest mass", named_a),
                         ("a group's neutral mass, nearest mass", named_b),
                         ("levels 1 and 2 of the rule in exercise 8", lv12)):
        pairs = [(n, t) for n, t in pairs if n >= 0]
        wrong = sum(n != t for n, t in pairs)
        print(f"{label:<56} {len(pairs):6d} {wrong:6d} {100 * wrong / max(len(pairs), 1):11.1f}%")


_demo("evidence ladder", _ladder_demo, needs=_UPTO[9])

The first row is what a bare "best match by mass" gives when no one has grouped anything. Each
later row adds something you wrote: the group, then the isotope pattern and the standard. Read
down the columns: how many names each row gives, and what share of them is wrong. Now take
evidence away from the full rule one piece at a time and see which level each piece is
responsible for.

In [ ]:
def _ablation_demo() -> None:
    lib = WORLD.library
    no_std = lib._replace(std_rt=np.full(len(lib.names), np.nan))
    print(f"{'evidence available':<34} {'level 1':>8} {'level 2':>8} {'level 3':>8} {'level 4':>8} {'wrong at 1-3':>13}")
    for label, kw in (("mass + pattern + standards", {}), ("no authentic standards", {"lib": no_std}),
                      ("no isotope pattern", {"drop_pattern": True}),
                      ("mass only", {"lib": no_std, "drop_pattern": True})):
        s = run_pipeline(WORLD, **kw).scores
        print(f"{label:<34} " + " ".join(f"{s[k][0]:8d}" for k in (1, 2, 3, 4))
              + f" {sum(s[k][1] for k in (1, 2, 3)):13d}")


_demo("evidence taken away", _ablation_demo, needs=_UPTO[9])

## 11. How much of that was one world?

Everything above is one seed. The same chain, run on the next several seeds, shows which of the
findings were the world and which were the method. This cell prints per-world counts and then
totals them; a rate from a few groups is a rough thing, so the totals are the figure to read.

In [ ]:
def _seed_spread() -> None:
    worlds = 6
    totals = {k: [0, 0] for k in (1, 2, 3, 4)}
    print(f"{'seed':>10} {'groups':>7} " + " ".join(f"{'L' + str(k) + ' n/wrong':>11}" for k in (1, 2, 3)) + f" {'L4 n':>6}")
    some_wrong_at_2 = 0
    for s in range(SEED, SEED + worlds):
        w = WORLD if s == SEED else generate_world(s)
        p = run_pipeline(w)
        for k in totals:
            totals[k][0] += p.scores[k][0]
            totals[k][1] += p.scores[k][1]
        some_wrong_at_2 += p.scores[2][1] > 0
        print(f"{s:10d} {len(p.groups.M):7d} "
              + " ".join(f"{p.scores[k][0]:7d}/{p.scores[k][1]:<3d}" for k in (1, 2, 3)) + f" {p.scores[4][0]:6d}")
    print(f"\ntotals over {worlds} worlds:")
    for k in (1, 2, 3):
        n, wrong = totals[k]
        print(f"  level {k}: {n} groups, {wrong} wrong ({100 * wrong / max(n, 1):.1f} %)")
    print(f"  level 4: {totals[4][0]} groups with no claim")
    print(f"worlds with at least one wrong level-2 name: {some_wrong_at_2} of {worlds}")


_demo("seed spread", _seed_spread, needs=_UPTO[9], limit=300)

## 12. Self-check

1. A feature list straight from peak picking has more entries than the sample has compounds. What
   explains most of the excess in a clean sample?
   - (a) most of the entries are noise, so a cleaner instrument would give one entry per compound
   - (b) one compound shows several adduct ions, and each ion shows isotope peaks, and each of them is a
     feature until something groups them
   - (c) the retention-time axes of the runs were not lined up, which makes every feature appear twice

2. A group of one ion is read as `[M+H]+`. When is the neutral mass that this gives wrong?
   - (a) never: `[M+H]+` is the only adduct that can occur
   - (b) only when the library does not contain the compound
   - (c) when the ion is really another adduct, such as the sodium one: the mass is out by the
     difference between the two adducts, which can match a different compound exactly

3. A group is reported at level 2: exactly one library entry matches its mass and its isotope
   pattern, and no standard was run. Which is a reason it can be wrong without any bug in the code?
   - (a) another compound with the same formula has the same mass and the same pattern, and nothing in
     the evidence separates them
   - (b) isotope patterns depend on how old the instrument is
   - (c) level 2 is defined as needing a standard, so it cannot be reached without one

4. Suppose retention times drift between runs by more than a peak is wide. Why is warping better than
   simply matching features with a much wider retention-time window?
   - (a) a wide window is slower to compute, and that is the only difference
   - (b) a wide window can only find more of the same ion, so it is always the better choice
   - (c) a wide window also pairs features that are different ions, so the table gets false merges as well
     as recall, while a warp moves the same ions onto the same time

5. In one world a level has no wrong group among a few dozen. What may you conclude?
   - (a) the level is always right
   - (b) that in this world, with this library and these standards, none of those groups was wrong; the
     count is small, other worlds may differ, and it says nothing about compounds without a standard
   - (c) the standards are more accurate than the instrument

Answers, with the reasoning, come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
def _methods_statement() -> None:
    p = run_pipeline(WORLD)
    c, g = p.consensus, p.groups
    print("METHODS — an annotated feature list without these lines cannot be reproduced or compared")
    print(f"  data              {len(WORLD.runs)} runs of {N_SCANS} scans, centroided peaks; noise level = median peak intensity of each run")
    print(f"  mass traces       peaks clustered by m/z, gap at most {CLUSTER_PPM:g} ppm, traces seen in at least {MIN_TRACE_POINTS} scans")
    print(f"  peaks             moving average of {SMOOTH_WINDOW} scans; signal-to-noise at least {SNR:g}; "
          f"prominence at least {PROM_FRAC:g} of height within {BASE_SCANS} scans")
    print(f"  alignment         piecewise-linear warp to run 0 through {N_KNOTS} median anchors; features matched within "
          f"{MATCH_PPM:g} ppm and {MATCH_RT_TOL:g} min; kept if found in at least {MIN_RUNS} runs")
    print(f"  grouping          isotope peaks and {len(ADDUCTS)} adducts ({', '.join(ADDUCT_NAMES)}), within {GROUP_PPM:g} ppm and {GROUP_RT_TOL:g} min; "
          "a lone ion is read as [M+H]+")
    print(f"  annotation        library of {len(WORLD.library.names)} entries; mass window {ANNOT_PPM:g} ppm, standard retention time "
          f"window {ANNOT_RT_TOL:g} min, M+1 / M ratio within {100 * ISO_TOL:g} % of the carbon count's expectation")
    print(f"  levels            1: library mass and an authentic standard's retention time; 2: mass and isotope pattern; 3: a class; 4: nothing claimed "
          "(the lesson's own rule, in the form of the Metabolomics Standards Initiative's levels)")
    print(f"  result            {len(c.mz)} features -> {len(g.M)} groups, reported as "
          + ", ".join(f"{p.scores[k][0]} at level {k}" for k in (1, 2, 3, 4))
          + f"; wrong at levels 1 to 3: {sum(p.scores[k][1] for k in (1, 2, 3))} (against the planted compounds)")


_demo("methods statement", _methods_statement, needs=_UPTO[9])

## What you built, and where it goes next

You now have the chain from raw peaks to an annotated feature list, and the habit it teaches: a
name on a feature is a claim, and the claim comes with a level that says what evidence stands
behind it. The measurements to carry forward are the ones this notebook printed: what each
step did to the number of features, what the retention-time warp bought, how often an adduct
read as a molecule gave a wrong neutral mass, and how the false-annotation rate moved as evidence
was added or taken away. This lesson ran on generated scans so that it fits a laptop and has an
answer key; running it on a public metabolomics study is the capstone's business, and a study's
own licence has to be read from its record before any of it is used.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_cluster),
                              ("exercise 2", _check_detect),
                              ("exercise 3", _check_match),
                              ("exercise 4", _check_warp),
                              ("exercise 5", _check_isotopes),
                              ("exercise 6", _check_adducts),
                              ("exercise 7", _check_neutral),
                              ("exercise 8", _check_annotate),
                              ("exercise 9", _check_scoring)):
            _try(_name, _check)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.